In [ ]:
https://docs.wiseoldman.net/api/players/player-endpoints

In [ ]:
import random, time, requests, pandas as pd
from io import StringIO
from datetime import datetime, timezone, timedelta

HEADERS = {"User-Agent": "WebDataScience project - simon.spillane@colorado.edu"}
HS_URL = "https://secure.runescape.com/m=hiscore_oldschool/overall"
BASE = "https://api.wiseoldman.net/v2"
TARGET = datetime(2021, 12, 9, tzinfo=timezone.utc)
WINDOW = timedelta(days=30)

def random_names(n_pages=200, max_rank=2_000_000, pause=2.0, seed=42):
    random.seed(seed)
    pages = random.sample(range(1, max_rank // 25 + 1), n_pages)
    names = []
    for p in pages:
        r = requests.get(HS_URL, params={"table": 0, "page": p}, headers=HEADERS, timeout=30)
        r.raise_for_status()
        tbl = max(pd.read_html(StringIO(r.text)), key=len)
        col = [c for c in tbl.columns if "name" in str(c).lower()][0]
        batch = tbl[col].astype(str).str.replace("\xa0", " ").str.strip().tolist()
        names.append(random.choice(batch))      # one name per page = cleaner random sample
        time.sleep(pause)
    return names

def wom_get(url, params=None):
    while True:
        r = requests.get(url, params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429:
            time.sleep(int(r.headers.get("Retry-After", 30))); continue
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()

def baseline_and_latest(u):
    p = wom_get(f"{BASE}/players/{u}")
    time.sleep(1)
    if not p or p["type"] != "regular" or not p.get("latestSnapshot"):
        return None
    snaps = wom_get(f"{BASE}/players/{u}/snapshots",
                    {"startDate": (TARGET - WINDOW).isoformat().replace("+00:00", "Z"),
                     "endDate":   (TARGET + WINDOW).isoformat().replace("+00:00", "Z"),
                     "limit": 50})
    time.sleep(1)
    if not snaps:
        return None
    t = lambda s: datetime.fromisoformat(s["createdAt"].replace("Z", "+00:00"))
    base = min(snaps, key=lambda s: abs(t(s) - TARGET))
    latest = p["latestSnapshot"]
    days = (t(latest) - t(base)).days
    if days <= 0:
        return None
    kc = lambda s: {b: max(v["kills"], 0) for b, v in s["data"]["bosses"].items()}
    b, l = kc(base), kc(latest)
    return {boss: (l[boss] - b.get(boss, 0)) / days for boss in l}   # kills/day

In [2]:
import json, math, os, random, time
from io import StringIO

import numpy as np
import pandas as pd
import requests

# ---------------- config ----------------
HEADERS = {"User-Agent": "WebDataScience project - simon.spillane@colorado.edu"}
HS_PAGE = "https://secure.runescape.com/m=hiscore_oldschool/overall"
HS_JSON = "https://secure.runescape.com/m=hiscore_oldschool/index_lite.json"

MAX_RANK = 2_000_000
STRATA = [(1, 10_000), (10_001, 100_000), (100_001, 1_000_000), (1_000_001, MAX_RANK)]
N_PER_STRATUM = 75            # 4 strata x 75 = 300 accounts; raise this to scale up
PAUSE = 2.0                   # seconds between requests
SEED = 42
CKPT = "boss_sample.jsonl"    # one JSON object per sampled account

# activities in index_lite that aren't bosses
NON_BOSS_PREFIXES = ("Clue Scrolls", "League Points", "Deadman Points", "Bounty Hunter",
                     "LMS", "PvP Arena", "Soul Wars", "Rifts closed", "Colosseum Glory",
                     "Collections Logged")

def is_boss(name):
    return not name.startswith(NON_BOSS_PREFIXES)

# ---------------- http ----------------
session = requests.Session()
session.headers.update(HEADERS)

def get(url, params=None):
    for attempt in range(6):
        r = session.get(url, params=params, timeout=30)
        if r.status_code == 404:
            return None
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(int(r.headers.get("Retry-After", 30 * (attempt + 1))))
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"gave up on {url} {params}")

def name_at_rank(rank):
    """Return the player name at an overall hiscores rank (None if the row is missing)."""
    page = (rank - 1) // 25 + 1
    row = (rank - 1) % 25
    r = get(HS_PAGE, {"table": 0, "page": page})
    if r is None:
        return None
    tbl = max(pd.read_html(StringIO(r.text)), key=len)
    col = [c for c in tbl.columns if "name" in str(c).lower()][0]
    names = tbl[col].astype(str).str.replace("\xa0", " ").str.strip().tolist()
    return names[row] if row < len(names) else None

def boss_kills(name):
    """Return {boss: kills} for a player, or None if the lookup 404s."""
    r = get(HS_JSON, {"player": name})
    if r is None:
        return None
    data = r.json()
    # score of -1 means unranked on that boss -> treat as 0 kills
    return {a["name"]: max(a["score"], 0) for a in data["activities"] if is_boss(a["name"])}

# ---------------- sampling plan ----------------
def make_plan():
    """Deterministic list of (stratum_index, rank) so a resumed run draws the same ranks."""
    rng = random.Random(SEED)
    plan = []
    for h, (lo, hi) in enumerate(STRATA):
        ranks = rng.sample(range(lo, hi + 1), N_PER_STRATUM)
        plan.extend((h, r) for r in ranks)
    rng.shuffle(plan)               # interleave strata so a partial run is still balanced
    return plan

def load_done():
    done = set()
    if os.path.exists(CKPT):
        with open(CKPT) as f:
            for line in f:
                rec = json.loads(line)
                done.add((rec["stratum"], rec["rank"]))
    return done

def collect():
    plan, done = make_plan(), load_done()
    todo = [p for p in plan if p not in done]
    print(f"{len(done)} already done, {len(todo)} to go")
    with open(CKPT, "a") as f:
        for i, (h, rank) in enumerate(todo, 1):
            name = name_at_rank(rank)
            time.sleep(PAUSE)
            kills = boss_kills(name) if name else None
            time.sleep(PAUSE)
            rec = {"stratum": h, "rank": rank, "name": name, "kills": kills}
            f.write(json.dumps(rec) + "\n")
            f.flush()
            if i % 10 == 0:
                print(f"{i}/{len(todo)}")

# ---------------- estimation ----------------
def estimate():
    recs = [json.loads(l) for l in open(CKPT)]
    ok = [r for r in recs if r["kills"]]
    print(f"{len(ok)} usable of {len(recs)} sampled "
          f"({len(recs) - len(ok)} failed lookups)")

    df = pd.DataFrame([{"stratum": r["stratum"], **r["kills"]} for r in ok]).fillna(0)
    bosses = [c for c in df.columns if c != "stratum"]

    rows = {}
    for b in bosses:
        total, var = 0.0, 0.0
        for h, (lo, hi) in enumerate(STRATA):
            x = df.loc[df.stratum == h, b].to_numpy(float)
            n, N = len(x), hi - lo + 1
            if n < 2:
                continue
            total += N * x.mean()
            var += N**2 * x.var(ddof=1) / n * (1 - n / N)   # finite-population correction
        se = math.sqrt(var)
        rows[b] = {"est_total_kills": total, "se": se,
                   "ci95_low": max(total - 1.96 * se, 0), "ci95_high": total + 1.96 * se}
    out = pd.DataFrame(rows).T.sort_values("est_total_kills", ascending=False)

    # share of each stratum that has any kills at all, useful context for the heavy tail
    frac = df.groupby("stratum")[bosses].apply(lambda g: (g > 0).mean()).T
    frac.columns = [f"frac_{STRATA[h][0]}-{STRATA[h][1]}" for h in frac.columns]
    return out, frac

if __name__ == "__main__":
    collect()
    totals, frac = estimate()
    pd.set_option("display.float_format", "{:,.0f}".format)
    print(totals.head(20))
    totals.to_csv("boss_kill_estimates.csv")

0 already done, 300 to go
10/300
20/300
30/300
40/300
50/300
60/300
70/300
80/300
90/300
100/300
110/300
120/300
130/300
140/300
150/300
160/300
170/300
180/300
190/300
200/300
210/300
220/300
230/300
240/300
250/300
260/300
270/300
280/300
290/300
300/300
0 usable of 300 sampled (300 failed lookups)


KeyError: 'est_total_kills'

In [3]:
import os
if os.path.exists("boss_sample.jsonl"):
    os.remove("boss_sample.jsonl")

# 1) does the rankings page parse?
r = session.get(HS_PAGE, params={"table": 0, "page": 1}, timeout=30)
print("rankings page:", r.status_code, r.url)
print("name at rank 1:", name_at_rank(1))
print("name at rank 7:", name_at_rank(7))

# 2) does the player endpoint work for that name?
n = name_at_rank(1)
r = session.get(HS_JSON, params={"player": n}, timeout=30)
print("player lookup:", r.status_code, r.url)
print(r.text[:300])

rankings page: 200 https://secure.runescape.com/m=hiscore_oldschool/overall?table=0&page=1
name at rank 1: nan
name at rank 7: nan
player lookup: 404 https://secure.runescape.com/m=hiscore_oldschool/index_lite.json?player=nan
<!DOCTYPE html><html><head><meta charset="UTF-8"/><title>404 - Page not found</title><meta http-equiv="refresh" content="0;url=https://www.runescape.com/error404"/></head></html>
<script>window.__CF$cv$params={r:'a5a2cf019f398b71',t:'MTc5NDYyNTY3NA==',u:'01a1d806a25073a186ea1024f1e2cedb',ut:'M5DV97h


In [4]:
from bs4 import BeautifulSoup   # you already use this for the Aurora scraper

r = session.get(HS_PAGE, params={"table": 0, "page": 1}, timeout=30)
soup = BeautifulSoup(r.text, "lxml")
rows = [tr for tr in soup.select("tr") if tr.find_all("td")]
print(len(rows), "data rows")
print(rows[0].prettify()[:1500])

26 data rows
<tr>
 <td width="16">
 </td>
 <th class="right">
  Rank
 </th>
 <th class="left" style="min-width: 100px;">
  Name
 </th>
 <th class="left">
  Level
 </th>
 <th class="right">
  XP
 </th>
</tr>



In [6]:
from bs4 import BeautifulSoup

def names_on_page(page):
    r = get(HS_PAGE, {"table": 0, "page": page})
    if r is None:
        return []
    soup = BeautifulSoup(r.text, "lxml")
    names = []
    for tr in soup.select("tr"):
        if len(tr.find_all("td")) < 3:
            continue
        # name = first link in the row whose text isn't just a number
        for a in tr.find_all("a"):
            t = a.get_text(strip=True).replace("\xa0", " ")
            if t and not t.replace(",", "").isdigit():
                names.append(t)
                break
    return names

def name_at_rank(rank):
    names = names_on_page((rank - 1) // 25 + 1)
    row = (rank - 1) % 25
    if len(names) < 25 and row >= len(names):
        return None
    name = names[row]
    if not name or name.lower() == "nan":
        raise RuntimeError(f"bad name parsed at rank {rank}: {name!r}")
    return name

In [7]:
print(len(names_on_page(1)), names_on_page(1)[:5])   # expect 25 names
n = name_at_rank(1)
r = session.get(HS_JSON, params={"player": n}, timeout=30)
print(n, r.status_code)
print(r.text[:300])


25 ['FasT 07', 'City Morgue', 'ne cro', 'Enjoys', 'he box jonge']
FasT 07 200
{"name":"FasT 07","skills":[{"id":0,"name":"Overall","rank":1,"level":2376,"xp":4800000000},{"id":1,"name":"Attack","rank":316,"level":99,"xp":200000000},{"id":2,"name":"Defence","rank":251,"level":99,"xp":200000000},{"id":3,"name":"Strength","rank":567,"level":99,"xp":200000000},{"id":4,"name":"Hit


In [9]:
import json
data = session.get(HS_JSON, params={"player": "FasT 07"}, timeout=30).json()
print(data.keys())
print(len(data["activities"]), "activities")
for a in data["activities"][:6]:
    print(a)
print([a["name"] for a in data["activities"] if not is_boss(a["name"])])
print(boss_kills("FasT 07"))

dict_keys(['name', 'skills', 'activities'])
92 activities
{'id': 0, 'name': 'Grid Points', 'rank': -1, 'score': 0}
{'id': 1, 'name': 'League Points', 'rank': -1, 'score': 0}
{'id': 2, 'name': 'Deadman Points', 'rank': -1, 'score': 0}
{'id': 3, 'name': 'Bounty Hunter - Hunter', 'rank': -1, 'score': 0}
{'id': 4, 'name': 'Bounty Hunter - Rogue', 'rank': -1, 'score': 0}
{'id': 5, 'name': 'Bounty Hunter (Legacy) - Hunter', 'rank': -1, 'score': 0}
['League Points', 'Deadman Points', 'Bounty Hunter - Hunter', 'Bounty Hunter - Rogue', 'Bounty Hunter (Legacy) - Hunter', 'Bounty Hunter (Legacy) - Rogue', 'Clue Scrolls (all)', 'Clue Scrolls (beginner)', 'Clue Scrolls (easy)', 'Clue Scrolls (medium)', 'Clue Scrolls (hard)', 'Clue Scrolls (elite)', 'Clue Scrolls (master)', 'LMS - Rank', 'PvP Arena - Rank', 'Soul Wars Zeal', 'Rifts closed', 'Colosseum Glory', 'Collections Logged']
{'Grid Points': 0, 'Abyssal Sire': 2564, 'Alchemical Hydra': 4352, 'Amoxliatl': 564, 'Araxxor': 1594, 'Artio': 4828, 'Ba

In [8]:
if os.path.exists("boss_sample.jsonl"):
    os.remove("boss_sample.jsonl")     # clear any bad records from the failed run

N_PER_STRATUM = 3                      # 12 accounts, about a minute
collect()
totals, frac = estimate()
print(totals.head(10))

0 already done, 12 to go
10/12
12 usable of 12 sampled (0 failed lookups)
                        est_total_kills            se      ci95_low  \
Kraken                     3.554167e+08  1.793172e+08  3.954919e+06   
Zulrah                     2.944867e+08  1.297985e+08  4.008156e+07   
The Corrupted Gauntlet     2.243667e+08  1.155339e+08  0.000000e+00   
The Royal Titans           2.028400e+08  1.624833e+08  0.000000e+00   
Vardorvis                  1.983233e+08  1.342445e+08  0.000000e+00   
Wintertodt                 1.920600e+08  8.394316e+07  2.753141e+07   
Spindel                    1.910800e+08  9.472736e+07  5.414371e+06   
Dagannoth Rex              1.782000e+08  8.839393e+07  4.947907e+06   
Cerberus                   1.356700e+08  5.671143e+07  2.451560e+07   
Dagannoth Prime            1.328000e+08  7.668507e+07  0.000000e+00   

                           ci95_high  
Kraken                  7.068784e+08  
Zulrah                  5.488918e+08  
The Corrupted Gauntlet  4.5

In [11]:
NON_BOSS_PREFIXES = ("Clue Scrolls", "League Points", "Deadman Points", "Bounty Hunter",
                     "LMS", "PvP Arena", "Soul Wars", "Rifts closed", "Colosseum Glory",
                     "Collections Logged", "Grid Points")

In [12]:
if os.path.exists("boss_sample.jsonl"):
    os.remove("boss_sample.jsonl")     # the rank plan changes with sample size

N_PER_STRATUM = 75
collect()                              # ~20 minutes; resumable if interrupted
totals, frac = estimate()
pd.set_option("display.float_format", "{:,.0f}".format)   # avoids scientific notation
print(totals.head(20))
totals.to_csv("boss_kill_estimates.csv")

0 already done, 300 to go
10/300
20/300
30/300
40/300
50/300
60/300
70/300
80/300
90/300
100/300
110/300
120/300
130/300
140/300
150/300
160/300
170/300
180/300
190/300
200/300
210/300
220/300
230/300
240/300
250/300
260/300
270/300
280/300
290/300
300/300
298 usable of 300 sampled (2 failed lookups)
                           est_total_kills          se    ci95_low  \
Kraken                       1,030,899,679 139,866,687 756,760,973   
Zulrah                         566,102,670 116,202,554 338,345,665   
Alchemical Hydra               338,734,734  67,903,786 205,643,314   
Cerberus                       327,333,310  57,005,596 215,602,342   
Wintertodt                     303,581,470  26,980,740 250,699,219   
Thermonuclear Smoke Devil      298,790,208  49,849,996 201,084,215   
Vorkath                        293,952,179  47,960,493 199,949,612   
Barrows Chests                 275,226,999  32,679,614 211,174,956   
Giant Mole                     259,873,220  51,434,521 159,061,559  

In [15]:
check = totals[["est_total_kills"]].join(frac)
pd.set_option("display.float_format", "{:,.2f}".format)
print(check.sort_values("est_total_kills", ascending=False).head(15))

                           est_total_kills  frac_1-10000  frac_10001-100000  \
Kraken                    1,030,899,678.54          1.00               0.99   
Zulrah                      566,102,670.32          0.99               0.97   
Alchemical Hydra            338,734,734.25          0.96               0.77   
Cerberus                    327,333,309.59          0.97               0.85   
Wintertodt                  303,581,470.32          1.00               1.00   
Thermonuclear Smoke Devil   298,790,208.22          0.99               0.99   
Vorkath                     293,952,179.00          0.97               0.96   
Barrows Chests              275,226,999.09          1.00               1.00   
Giant Mole                  259,873,220.09          0.99               1.00   
Brutus                      214,192,060.27          0.81               0.65   
Abyssal Sire                211,853,413.70          0.95               0.81   
Sarachnis                   193,497,629.22          

In [19]:
recs = [json.loads(l) for l in open(CKPT)]
ok = [r for r in recs if r["kills"]]
df = pd.DataFrame([{"stratum": r["stratum"], **r["kills"]} for r in ok]).fillna(0)

post = [b for b in POST_2021 if b in df.columns]
missing = sorted(POST_2021 - set(post))
print("not found in data:", missing)

df["active_post"] = (df[post]sum(axis=1) > 0).astype(float)

def strat_total(x):
    """Stratified total and SE for a per-account column x (same index as df)."""
    total, var = 0.0, 0.0
    for h, (lo, hi) in enumerate(STRATA):
        v = x[df.stratum == h].to_numpy(float)
        n, N = len(v), hi - lo + 1
        if n < 2:
            continue
        total += N * v.mean()
        var += N**2 * v.var(ddof=1) / n * (1 - n / N)
    return total, math.sqrt(var)

# 1) how many ranked accounts are active on post-2021 content, and how many kills was that
n_act, se_act = strat_total(df["active_post"])
print(f"est. accounts with post-2021 boss kills: {n_act:,.0f} ± {1.96*se_act:,.0f}")

kills_post = df[post].sum(axis=1)
k_tot, k_se = strat_total(kills_post)
print(f"est. total kills on post-2021 bosses: {k_tot:,.0f} ± {1.96*k_se:,.0f}")

# 2) the same estimate per boss
per_boss = {b: strat_total(df[b]) for b in post}
print(pd.DataFrame(per_boss, index=["total", "se"]).T.sort_values("total", ascending=False))

# 3) among those active accounts, what are they doing on older bosses?
old = [b for b in df.columns if b not in post and b not in ("stratum", "active_post")]
for b in ["Zulrah", "Vorkath", "Kraken"]:
    num, _ = strat_total(df[b] * df["active_post"])
    print(b, f"kills by post-2021-active accounts: {num:,.0f} ({num/strat_total(df[b])[0]:.0%} of total)")

not found in data: []
est. accounts with post-2021 boss kills: 1,288,263 ± 144,659
est. total kills on post-2021 bosses: 1,786,634,287 ± 408,307,982
                                       total            se
Brutus                        214,192,060.27 43,867,721.83
Vardorvis                     176,867,658.45 46,589,251.07
Araxxor                       167,914,328.77 32,015,615.72
Artio                         133,040,392.69 79,602,373.64
The Leviathan                 126,181,505.02 60,961,350.46
Duke Sucellus                 121,297,163.47 24,180,753.54
Nex                           115,194,350.68 34,756,282.68
The Royal Titans               95,876,761.64 16,305,604.57
Lunar Chests                   86,353,600.00 12,989,774.17
Shellbane Gryphon              75,791,068.49 26,340,359.29
Phantom Muspah                 66,038,547.95 13,961,476.98
Scurrius                       58,862,305.02 13,599,536.65
The Hueycoatl                  54,635,497.72 14,232,480.75
Calvar'ion               

In [37]:
SEED = 99                      # any number you haven't used before (you've used 42 and 7)
CKPT = "boss_sample_c.jsonl"   # a new file, so it doesn't mix with samples A and B
collect()

0 already done, 1200 to go


KeyboardInterrupt: 

In [20]:
import json, math, os, random, time

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---------------- config ----------------
HEADERS = {"User-Agent": "WebDataScience project - simon.spillane@colorado.edu"}
HS_PAGE = "https://secure.runescape.com/m=hiscore_oldschool/overall"
HS_JSON = "https://secure.runescape.com/m=hiscore_oldschool/index_lite.json"

MAX_RANK = 2_000_000
STRATA = [(1, 10_000), (10_001, 100_000), (100_001, 1_000_000), (1_000_001, MAX_RANK)]
N_PER_STRATUM = 75            # 4 x 75 = 300 accounts
PAUSE = 2.0
SEED = 42
CKPT = "boss_sample.jsonl"

NON_BOSS_PREFIXES = ("Clue Scrolls", "League Points", "Deadman Points", "Bounty Hunter",
                     "LMS", "PvP Arena", "Soul Wars", "Rifts closed", "Colosseum Glory",
                     "Collections Logged", "Grid Points")

# bosses I believe released after 2021-12-09: VERIFY each on its wiki page
POST_2021 = {
    "Nex",                         # borderline, check the exact date against Dec 9, 2021
    "Phantom Muspah",
    "Tombs of Amascut", "Tombs of Amascut: Expert Mode",
    "Duke Sucellus", "The Leviathan", "Vardorvis", "The Whisperer",
    "Sol Heredit",
    "Scurrius", "Araxxor", "Brutus", "Amoxliatl", "The Hueycoatl",
    "The Royal Titans", "Yama",
    "Artio", "Calvar'ion", "Spindel",
    "Shellbane Gryphon",
    "Mad Angel",
    "Lunar Chests",
    "Doom of Mokhaiotl",
    "The Fractured Archive",
    "Maggot King",
}

def is_boss(name):
    return not name.startswith(NON_BOSS_PREFIXES)

# ---------------- http ----------------
session = requests.Session()
session.headers.update(HEADERS)

def get(url, params=None):
    for attempt in range(6):
        r = session.get(url, params=params, timeout=30)
        if r.status_code == 404:
            return None
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(int(r.headers.get("Retry-After", 30 * (attempt + 1))))
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"gave up on {url} {params}")

def names_on_page(page):
    r = get(HS_PAGE, {"table": 0, "page": page})
    if r is None:
        return []
    soup = BeautifulSoup(r.text, "lxml")
    trs = soup.select("tr")
    name_idx = None
    for tr in trs:
        labels = [c.get_text(strip=True).lower() for c in tr.find_all(["td", "th"])]
        if "name" in labels:
            name_idx = labels.index("name")
            break
    if name_idx is None:
        raise RuntimeError("couldn't find a Name header on the rankings page")
    names = []
    for tr in trs:
        if tr.find("th"):
            continue
        cells = tr.find_all(["td", "th"])
        if len(cells) <= name_idx:
            continue
        t = cells[name_idx].get_text(" ", strip=True).replace("\xa0", " ").strip()
        if t:
            names.append(t)
    return names

def name_at_rank(rank):
    names = names_on_page((rank - 1) // 25 + 1)
    row = (rank - 1) % 25
    if row >= len(names):
        return None
    name = names[row]
    if not name or name.lower() == "nan":
        raise RuntimeError(f"bad name parsed at rank {rank}: {name!r}")
    return name

def boss_kills(name):
    r = session.get(HS_JSON, params={"player": name}, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"lookup for {name!r} -> HTTP {r.status_code} ({r.url})")
    data = r.json()
    # score of -1 = unranked on that boss -> treat as 0 kills
    return {a["name"]: max(a["score"], 0) for a in data["activities"] if is_boss(a["name"])}

# ---------------- sampling ----------------
N_BY_STRATUM = {0: 53, 1: 164, 2: 814, 3: 169}   # replace with your Neyman output

def make_plan():
    plan = []
    for h, (lo, hi) in enumerate(STRATA):
        rng = random.Random(SEED * 1000 + h)
        plan.extend((h, r) for r in rng.sample(range(lo, hi + 1), N_BY_STRATUM[h]))
    random.Random(SEED).shuffle(plan)
    return plan

def load_done():
    done = set()
    if os.path.exists(CKPT):
        with open(CKPT) as f:
            for line in f:
                rec = json.loads(line)
                done.add((rec["stratum"], rec["rank"]))
    return done

def collect():
    plan, done = make_plan(), load_done()
    todo = [p for p in plan if p not in done]
    print(f"{len(done)} already done, {len(todo)} to go")
    fails = 0
    with open(CKPT, "a") as f:
        for i, (h, rank) in enumerate(todo, 1):
            try:
                name = name_at_rank(rank)
                time.sleep(PAUSE)
                if not name:
                    raise RuntimeError(f"no name at rank {rank}")
                kills = boss_kills(name)
                time.sleep(PAUSE)
            except Exception as e:
                fails += 1
                print(f"skip rank {rank}: {e}")
                if fails >= 5:
                    raise RuntimeError("5 failures in a row, stopping")
                continue
            fails = 0
            f.write(json.dumps({"stratum": h, "rank": rank, "name": name, "kills": kills}) + "\n")
            f.flush()
            if i % 10 == 0:
                print(f"{i}/{len(todo)}")

# ---------------- estimation ----------------
def strat_total(df, x):
    """Stratified population total and SE for per-account column x."""
    total, var = 0.0, 0.0
    for h, (lo, hi) in enumerate(STRATA):
        v = x[df.stratum == h].to_numpy(float)
        n, N = len(v), hi - lo + 1
        if n < 2:
            continue
        total += N * v.mean()
        var += N**2 * v.var(ddof=1) / n * (1 - n / N)     # finite-population correction
    return total, math.sqrt(var)

def load_frame():
    recs = [json.loads(l) for l in open(CKPT)]
    ok = [r for r in recs if r["kills"]]
    if not ok:
        raise RuntimeError("no usable records in the checkpoint")
    print(f"{len(ok)} usable of {len(recs)} records")
    df = pd.DataFrame([{"stratum": r["stratum"], **r["kills"]} for r in ok]).fillna(0)
    return df

def estimate():
    df = load_frame()
    bosses = [c for c in df.columns if c != "stratum"]

    post = sorted(b for b in POST_2021 if b in df.columns)
    print("post-2021 bosses found in data:", post)
    print("listed but not in data (check spelling):", sorted(POST_2021 - set(post)))

    # per-boss totals
    rows = {}
    for b in bosses:
        t, se = strat_total(df, df[b])
        rows[b] = {"est_total_kills": t, "se": se,
                   "ci95_low": max(t - 1.96 * se, 0), "ci95_high": t + 1.96 * se,
                   "post_2021": b in post}
    totals = pd.DataFrame(rows).T.sort_values("est_total_kills", ascending=False)

    # share of each stratum with any kills
    frac = df.groupby("stratum")[bosses].apply(lambda g: (g > 0).mean()).T
    frac.columns = [f"frac_{STRATA[h][0]}-{STRATA[h][1]}" for h in frac.columns]

    # domain estimation: flag every account, never drop any, so the weights stay valid
    df["active_post"] = (df[post].sum(axis=1) > 0).astype(float)
    n_act, se_act = strat_total(df, df["active_post"])
    k_tot, k_se = strat_total(df, df[post].sum(axis=1))
    act_by_stratum = df.groupby("stratum")["active_post"].mean()
    act_by_stratum.index = [f"{STRATA[h][0]}-{STRATA[h][1]}" for h in act_by_stratum.index]

    summary = {
        "accounts_with_post2021_kills": (n_act, 1.96 * se_act),
        "total_post2021_boss_kills": (k_tot, 1.96 * k_se),
    }

    # share of each older boss's kills that come from post-2021-active accounts
    old = [b for b in bosses if b not in post]
    share = {}
    for b in old:
        whole, _ = strat_total(df, df[b])
        part, _ = strat_total(df, df[b] * df["active_post"])
        share[b] = part / whole if whole else float("nan")
    share = pd.Series(share).sort_values(ascending=False)

    return totals, frac, summary, act_by_stratum, share

if __name__ == "__main__":
    collect()
    totals, frac, summary, act_by_stratum, share = estimate()

    print("\n--- post-2021 activity ---")
    for k, (v, pm) in summary.items():
        print(f"{k}: {v:,.0f} ± {pm:,.0f}")
    print("\nfraction of sampled accounts with post-2021 boss kills, by stratum:")
    print(act_by_stratum.round(2))

    print("\n--- top 20 bosses by estimated total kills ---")
    t = totals.copy()
    for c in ["est_total_kills", "se", "ci95_low", "ci95_high"]:
        t[c] = t[c].astype(float).map("{:,.0f}".format)
    print(t.head(20))

    print("\n--- post-2021 bosses only ---")
    print(t[t.post_2021 == True])

    print("\nshare of older-boss kills from post-2021-active accounts (top 10):")
    print(share.head(10).round(2))

    totals.join(frac).to_csv("boss_kill_estimates.csv")

300 already done, 15 to go
10/15
313 usable of 315 records
post-2021 bosses found in data: ['Amoxliatl', 'Araxxor', 'Artio', 'Brutus', "Calvar'ion", 'Duke Sucellus', 'Lunar Chests', 'Mad Angel', 'Nex', 'Phantom Muspah', 'Scurrius', 'Shellbane Gryphon', 'Sol Heredit', 'Spindel', 'The Hueycoatl', 'The Leviathan', 'The Royal Titans', 'The Whisperer', 'Tombs of Amascut', 'Tombs of Amascut: Expert Mode', 'Vardorvis', 'Yama']
listed but not in data (check spelling): []

--- post-2021 activity ---
accounts_with_post2021_kills: 1,285,238 ± 140,587
total_post2021_boss_kills: 1,776,034,878 ± 388,053,110

fraction of sampled accounts with post-2021 boss kills, by stratum:
1-10000           0.99
10001-100000      0.96
100001-1000000    0.74
1000001-2000000   0.52
Name: active_post, dtype: float64

--- top 20 bosses by estimated total kills ---
                          est_total_kills           se     ci95_low  \
Kraken                      1,153,589,195  156,817,291  846,227,304   
Zulrah        

In [25]:
def estimates(ckpt):
    global CKPT
    CKPT = ckpt
    df = load_frame()
    bosses = [c for c in df.columns if c != "stratum"]
    return pd.DataFrame({b: strat_total(df, df[b]) for b in bosses}, index=["total", "se"]).T

a = estimates("boss_sample.jsonl")
b = estimates("boss_sample_b.jsonl")
z = (a.total - b.total) / ((a.se**2 + b.se**2) ** 0.5)
print(z.describe())
print("share with |z| > 2:", (z.abs() > 2).mean())   # should be small, roughly 5%
print(z[z.abs() > 2])

313 usable of 315 records
300 usable of 300 records
count   72.00
mean     0.16
std      1.00
min     -1.87
25%     -0.51
50%      0.23
75%      0.84
max      2.00
dtype: float64
share with |z| > 2: 0.0
Series([], dtype: float64)


In [26]:
recs = [json.loads(l) for l in open("boss_sample.jsonl")]
print(pd.Series([r["stratum"] for r in recs]).value_counts().sort_index())
keys = [(r["stratum"], r["rank"]) for r in recs]
print("duplicate ranks:", len(keys) - len(set(keys)))

0    75
1    80
2    80
3    80
Name: count, dtype: int64
duplicate ranks: 0


In [24]:
import json, math, os, random, time

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---------------- config ----------------
HEADERS = {"User-Agent": "WebDataScience project - simon.spillane@colorado.edu"}
HS_PAGE = "https://secure.runescape.com/m=hiscore_oldschool/overall"
HS_JSON = "https://secure.runescape.com/m=hiscore_oldschool/index_lite.json"

MAX_RANK = 2_000_000
STRATA = [(1, 10_000), (10_001, 100_000), (100_001, 1_000_000), (1_000_001, MAX_RANK)]
N_PER_STRATUM = 75            # 4 x 75 = 300 accounts
PAUSE = 2.0
SEED = 42
CKPT = "boss_sample_b.jsonl"

NON_BOSS_PREFIXES = ("Clue Scrolls", "League Points", "Deadman Points", "Bounty Hunter",
                     "LMS", "PvP Arena", "Soul Wars", "Rifts closed", "Colosseum Glory",
                     "Collections Logged", "Grid Points")

# bosses I believe released after 2021-12-09: VERIFY each on its wiki page
POST_2021 = {
    "Nex",                         # borderline, check the exact date against Dec 9, 2021
    "Phantom Muspah",
    "Tombs of Amascut", "Tombs of Amascut: Expert Mode",
    "Duke Sucellus", "The Leviathan", "Vardorvis", "The Whisperer",
    "Sol Heredit",
    "Scurrius", "Araxxor", "Brutus", "Amoxliatl", "The Hueycoatl",
    "The Royal Titans", "Yama",
    "Artio", "Calvar'ion", "Spindel",
    "Shellbane Gryphon",
    "Mad Angel",
    "Lunar Chests",
    "Doom of Mokhaiotl",
    "The Fractured Archive",
    "Maggot King",
}

def is_boss(name):
    return not name.startswith(NON_BOSS_PREFIXES)

# ---------------- http ----------------
session = requests.Session()
session.headers.update(HEADERS)

def get(url, params=None):
    for attempt in range(6):
        r = session.get(url, params=params, timeout=30)
        if r.status_code == 404:
            return None
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(int(r.headers.get("Retry-After", 30 * (attempt + 1))))
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"gave up on {url} {params}")

def names_on_page(page):
    r = get(HS_PAGE, {"table": 0, "page": page})
    if r is None:
        return []
    soup = BeautifulSoup(r.text, "lxml")
    trs = soup.select("tr")
    name_idx = None
    for tr in trs:
        labels = [c.get_text(strip=True).lower() for c in tr.find_all(["td", "th"])]
        if "name" in labels:
            name_idx = labels.index("name")
            break
    if name_idx is None:
        raise RuntimeError("couldn't find a Name header on the rankings page")
    names = []
    for tr in trs:
        if tr.find("th"):
            continue
        cells = tr.find_all(["td", "th"])
        if len(cells) <= name_idx:
            continue
        t = cells[name_idx].get_text(" ", strip=True).replace("\xa0", " ").strip()
        if t:
            names.append(t)
    return names

def name_at_rank(rank):
    names = names_on_page((rank - 1) // 25 + 1)
    row = (rank - 1) % 25
    if row >= len(names):
        return None
    name = names[row]
    if not name or name.lower() == "nan":
        raise RuntimeError(f"bad name parsed at rank {rank}: {name!r}")
    return name

def boss_kills(name):
    r = session.get(HS_JSON, params={"player": name}, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"lookup for {name!r} -> HTTP {r.status_code} ({r.url})")
    data = r.json()
    # score of -1 = unranked on that boss -> treat as 0 kills
    return {a["name"]: max(a["score"], 0) for a in data["activities"] if is_boss(a["name"])}

# ---------------- sampling ----------------
N_BY_STRATUM = {0: 53, 1: 164, 2: 814, 3: 169}   # replace with your Neyman output

def make_plan():
    plan = []
    for h, (lo, hi) in enumerate(STRATA):
        rng = random.Random(SEED * 1000 + h)
        plan.extend((h, r) for r in rng.sample(range(lo, hi + 1), N_BY_STRATUM[h]))
    random.Random(SEED).shuffle(plan)
    return plan

def load_done():
    done = set()
    if os.path.exists(CKPT):
        with open(CKPT) as f:
            for line in f:
                rec = json.loads(line)
                done.add((rec["stratum"], rec["rank"]))
    return done

def collect():
    plan, done = make_plan(), load_done()
    todo = [p for p in plan if p not in done]
    print(f"{len(done)} already done, {len(todo)} to go")
    fails = 0
    with open(CKPT, "a") as f:
        for i, (h, rank) in enumerate(todo, 1):
            try:
                name = name_at_rank(rank)
                time.sleep(PAUSE)
                if not name:
                    raise RuntimeError(f"no name at rank {rank}")
                kills = boss_kills(name)
                time.sleep(PAUSE)
            except Exception as e:
                fails += 1
                print(f"skip rank {rank}: {e}")
                if fails >= 5:
                    raise RuntimeError("5 failures in a row, stopping")
                continue
            fails = 0
            f.write(json.dumps({"stratum": h, "rank": rank, "name": name, "kills": kills}) + "\n")
            f.flush()
            if i % 10 == 0:
                print(f"{i}/{len(todo)}")

# ---------------- estimation ----------------
def strat_total(df, x):
    """Stratified population total and SE for per-account column x."""
    total, var = 0.0, 0.0
    for h, (lo, hi) in enumerate(STRATA):
        v = x[df.stratum == h].to_numpy(float)
        n, N = len(v), hi - lo + 1
        if n < 2:
            continue
        total += N * v.mean()
        var += N**2 * v.var(ddof=1) / n * (1 - n / N)     # finite-population correction
    return total, math.sqrt(var)

def load_frame():
    recs = [json.loads(l) for l in open(CKPT)]
    ok = [r for r in recs if r["kills"]]
    if not ok:
        raise RuntimeError("no usable records in the checkpoint")
    print(f"{len(ok)} usable of {len(recs)} records")
    df = pd.DataFrame([{"stratum": r["stratum"], **r["kills"]} for r in ok]).fillna(0)
    return df

def estimate():
    df = load_frame()
    bosses = [c for c in df.columns if c != "stratum"]

    post = sorted(b for b in POST_2021 if b in df.columns)
    print("post-2021 bosses found in data:", post)
    print("listed but not in data (check spelling):", sorted(POST_2021 - set(post)))

    # per-boss totals
    rows = {}
    for b in bosses:
        t, se = strat_total(df, df[b])
        rows[b] = {"est_total_kills": t, "se": se,
                   "ci95_low": max(t - 1.96 * se, 0), "ci95_high": t + 1.96 * se,
                   "post_2021": b in post}
    totals = pd.DataFrame(rows).T.sort_values("est_total_kills", ascending=False)

    # share of each stratum with any kills
    frac = df.groupby("stratum")[bosses].apply(lambda g: (g > 0).mean()).T
    frac.columns = [f"frac_{STRATA[h][0]}-{STRATA[h][1]}" for h in frac.columns]

    # domain estimation: flag every account, never drop any, so the weights stay valid
    df["active_post"] = (df[post].sum(axis=1) > 0).astype(float)
    n_act, se_act = strat_total(df, df["active_post"])
    k_tot, k_se = strat_total(df, df[post].sum(axis=1))
    act_by_stratum = df.groupby("stratum")["active_post"].mean()
    act_by_stratum.index = [f"{STRATA[h][0]}-{STRATA[h][1]}" for h in act_by_stratum.index]

    summary = {
        "accounts_with_post2021_kills": (n_act, 1.96 * se_act),
        "total_post2021_boss_kills": (k_tot, 1.96 * k_se),
    }

    # share of each older boss's kills that come from post-2021-active accounts
    old = [b for b in bosses if b not in post]
    share = {}
    for b in old:
        whole, _ = strat_total(df, df[b])
        part, _ = strat_total(df, df[b] * df["active_post"])
        share[b] = part / whole if whole else float("nan")
    share = pd.Series(share).sort_values(ascending=False)

    return totals, frac, summary, act_by_stratum, share

if __name__ == "__main__":
    collect()
    totals, frac, summary, act_by_stratum, share = estimate()

    print("\n--- post-2021 activity ---")
    for k, (v, pm) in summary.items():
        print(f"{k}: {v:,.0f} ± {pm:,.0f}")
    print("\nfraction of sampled accounts with post-2021 boss kills, by stratum:")
    print(act_by_stratum.round(2))

    print("\n--- top 20 bosses by estimated total kills ---")
    t = totals.copy()
    for c in ["est_total_kills", "se", "ci95_low", "ci95_high"]:
        t[c] = t[c].astype(float).map("{:,.0f}".format)
    print(t.head(20))

    print("\n--- post-2021 bosses only ---")
    print(t[t.post_2021 == True])

    print("\nshare of older-boss kills from post-2021-active accounts (top 10):")
    print(share.head(10).round(2))

    totals.join(frac).to_csv("boss_kill_estimates.csv")

0 already done, 300 to go
10/300
20/300
30/300
40/300
50/300
60/300
70/300
80/300
90/300
100/300
110/300
120/300
130/300
140/300
150/300
160/300
170/300
180/300
190/300
200/300
210/300
220/300
230/300
240/300
250/300
260/300
270/300
280/300
290/300
300/300
300 usable of 300 records
post-2021 bosses found in data: ['Amoxliatl', 'Araxxor', 'Artio', 'Brutus', "Calvar'ion", 'Doom of Mokhaiotl', 'Duke Sucellus', 'Lunar Chests', 'Mad Angel', 'Maggot King', 'Nex', 'Phantom Muspah', 'Scurrius', 'Shellbane Gryphon', 'Sol Heredit', 'Spindel', 'The Fractured Archive', 'The Hueycoatl', 'The Leviathan', 'The Royal Titans', 'The Whisperer', 'Tombs of Amascut', 'Tombs of Amascut: Expert Mode', 'Vardorvis', 'Yama']
listed but not in data (check spelling): []

--- post-2021 activity ---
accounts_with_post2021_kills: 1,198,800 ± 144,977
total_post2021_boss_kills: 1,853,786,933 ± 546,262,160

fraction of sampled accounts with post-2021 boss kills, by stratum:
1-10000           1.00
10001-100000      0.99

In [27]:
seen, kept = set(), []
for fn in ("boss_sample.jsonl", "boss_sample_b.jsonl"):
    for line in open(fn):
        rec = json.loads(line)
        key = (rec["stratum"], rec["rank"])
        if not rec["kills"] or key in seen:
            continue
        seen.add(key)
        kept.append(line)

with open("boss_sample_pooled.jsonl", "w") as f:
    f.writelines(kept)
print(len(kept), "pooled records")

CKPT = "boss_sample_pooled.jsonl"
totals, frac, summary, act_by_stratum, share = estimate()

315 pooled records
315 usable of 315 records
post-2021 bosses found in data: ['Amoxliatl', 'Araxxor', 'Artio', 'Brutus', "Calvar'ion", 'Doom of Mokhaiotl', 'Duke Sucellus', 'Lunar Chests', 'Mad Angel', 'Maggot King', 'Nex', 'Phantom Muspah', 'Scurrius', 'Shellbane Gryphon', 'Sol Heredit', 'Spindel', 'The Fractured Archive', 'The Hueycoatl', 'The Leviathan', 'The Royal Titans', 'The Whisperer', 'Tombs of Amascut', 'Tombs of Amascut: Expert Mode', 'Vardorvis', 'Yama']
listed but not in data (check spelling): []


In [30]:
N_PER_STRATUM, SEED, CKPT = 80, 42, "boss_sample.jsonl"
collect()          # skips everything done, retries only the missing ranks

374 already done, 0 to go


In [31]:
def make_plan():
    """Prefix-stable: each stratum has its own RNG, so raising N_PER_STRATUM only appends."""
    plan = []
    for h, (lo, hi) in enumerate(STRATA):
        rng = random.Random(SEED * 1000 + h)
        plan.extend((h, r) for r in rng.sample(range(lo, hi + 1), N_PER_STRATUM))
    random.Random(SEED).shuffle(plan)
    return plan

In [33]:
def neyman(df, col, total_n):
    """Accounts per stratum for a target column, proportional to N_h * S_h."""
    w = {}
    for h, (lo, hi) in enumerate(STRATA):
        v = df.loc[df.stratum == h, col].astype(float)
        w[h] = (hi - lo + 1) * v.std(ddof=1)
    s = sum(w.values())
    return {h: max(10, round(total_n * w[h] / s)) for h in w}   # floor of 10 per stratum

df = load_frame()                       # pooled checkpoint
df["post_kills"] = df[post].sum(axis=1)
print(neyman(df, "post_kills", 1200))

372 usable of 374 records
{0: 53, 1: 164, 2: 814, 3: 169}


In [39]:
import json, math, os, random, time

import pandas as pd
import requests
from bs4 import BeautifulSoup

# ---------------- config ----------------
HEADERS = {"User-Agent": "WebDataScience project - simon.spillane@colorado.edu"}
HS_PAGE = "https://secure.runescape.com/m=hiscore_oldschool/overall"
HS_JSON = "https://secure.runescape.com/m=hiscore_oldschool/index_lite.json"

MAX_RANK = 2_000_000
STRATA = [(1, 10_000), (10_001, 100_000), (100_001, 1_000_000), (1_000_001, MAX_RANK)]
N_PER_STRATUM = 75            # 4 x 75 = 300 accounts
PAUSE = 2.0
SEED = 98
CKPT = "boss_sample_c.jsonl"

NON_BOSS_PREFIXES = ("Clue Scrolls", "League Points", "Deadman Points", "Bounty Hunter",
                     "LMS", "PvP Arena", "Soul Wars", "Rifts closed", "Colosseum Glory",
                     "Collections Logged", "Grid Points")

# bosses I believe released after 2021-12-09: VERIFY each on its wiki page
POST_2021 = {
    "Nex",                         # borderline, check the exact date against Dec 9, 2021
    "Phantom Muspah",
    "Tombs of Amascut", "Tombs of Amascut: Expert Mode",
    "Duke Sucellus", "The Leviathan", "Vardorvis", "The Whisperer",
    "Sol Heredit",
    "Scurrius", "Araxxor", "Brutus", "Amoxliatl", "The Hueycoatl",
    "The Royal Titans", "Yama",
    "Artio", "Calvar'ion", "Spindel",
    "Shellbane Gryphon",
    "Mad Angel",
    "Lunar Chests",
    "Doom of Mokhaiotl",
    "The Fractured Archive",
    "Maggot King",
}

def is_boss(name):
    return not name.startswith(NON_BOSS_PREFIXES)

# ---------------- http ----------------
session = requests.Session()
session.headers.update(HEADERS)

def get(url, params=None):
    for attempt in range(6):
        r = session.get(url, params=params, timeout=30)
        if r.status_code == 404:
            return None
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(int(r.headers.get("Retry-After", 30 * (attempt + 1))))
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"gave up on {url} {params}")

def names_on_page(page):
    r = get(HS_PAGE, {"table": 0, "page": page})
    if r is None:
        return []
    soup = BeautifulSoup(r.text, "lxml")
    trs = soup.select("tr")
    name_idx = None
    for tr in trs:
        labels = [c.get_text(strip=True).lower() for c in tr.find_all(["td", "th"])]
        if "name" in labels:
            name_idx = labels.index("name")
            break
    if name_idx is None:
        raise RuntimeError("couldn't find a Name header on the rankings page")
    names = []
    for tr in trs:
        if tr.find("th"):
            continue
        cells = tr.find_all(["td", "th"])
        if len(cells) <= name_idx:
            continue
        t = cells[name_idx].get_text(" ", strip=True).replace("\xa0", " ").strip()
        if t:
            names.append(t)
    return names

def name_at_rank(rank):
    names = names_on_page((rank - 1) // 25 + 1)
    row = (rank - 1) % 25
    if row >= len(names):
        return None
    name = names[row]
    if not name or name.lower() == "nan":
        raise RuntimeError(f"bad name parsed at rank {rank}: {name!r}")
    return name

def boss_kills(name):
    r = session.get(HS_JSON, params={"player": name}, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f"lookup for {name!r} -> HTTP {r.status_code} ({r.url})")
    data = r.json()
    # score of -1 = unranked on that boss -> treat as 0 kills
    return {a["name"]: max(a["score"], 0) for a in data["activities"] if is_boss(a["name"])}

# ---------------- sampling ----------------
N_BY_STRATUM = {0: 53, 1: 164, 2: 814, 3: 169}   # replace with your Neyman output

def make_plan():
    plan = []
    for h, (lo, hi) in enumerate(STRATA):
        rng = random.Random(SEED * 1000 + h)
        plan.extend((h, r) for r in rng.sample(range(lo, hi + 1), N_BY_STRATUM[h]))
    random.Random(SEED).shuffle(plan)
    return plan

def load_done():
    done = set()
    if os.path.exists(CKPT):
        with open(CKPT) as f:
            for line in f:
                rec = json.loads(line)
                done.add((rec["stratum"], rec["rank"]))
    return done

def collect():
    plan, done = make_plan(), load_done()
    todo = [p for p in plan if p not in done]
    print(f"{len(done)} already done, {len(todo)} to go")
    fails = 0
    with open(CKPT, "a") as f:
        for i, (h, rank) in enumerate(todo, 1):
            try:
                name = name_at_rank(rank)
                time.sleep(PAUSE)
                if not name:
                    raise RuntimeError(f"no name at rank {rank}")
                kills = boss_kills(name)
                time.sleep(PAUSE)
            except Exception as e:
                fails += 1
                print(f"skip rank {rank}: {e}")
                if fails >= 5:
                    raise RuntimeError("5 failures in a row, stopping")
                continue
            fails = 0
            f.write(json.dumps({"stratum": h, "rank": rank, "name": name, "kills": kills}) + "\n")
            f.flush()
            if i % 10 == 0:
                print(f"{i}/{len(todo)}")

# ---------------- estimation ----------------
def strat_total(df, x):
    """Stratified population total and SE for per-account column x."""
    total, var = 0.0, 0.0
    for h, (lo, hi) in enumerate(STRATA):
        v = x[df.stratum == h].to_numpy(float)
        n, N = len(v), hi - lo + 1
        if n < 2:
            continue
        total += N * v.mean()
        var += N**2 * v.var(ddof=1) / n * (1 - n / N)     # finite-population correction
    return total, math.sqrt(var)

def load_frame():
    recs = [json.loads(l) for l in open(CKPT)]
    ok = [r for r in recs if r["kills"]]
    if not ok:
        raise RuntimeError("no usable records in the checkpoint")
    print(f"{len(ok)} usable of {len(recs)} records")
    df = pd.DataFrame([{"stratum": r["stratum"], **r["kills"]} for r in ok]).fillna(0)
    return df

def estimate():
    df = load_frame()
    bosses = [c for c in df.columns if c != "stratum"]

    post = sorted(b for b in POST_2021 if b in df.columns)
    print("post-2021 bosses found in data:", post)
    print("listed but not in data (check spelling):", sorted(POST_2021 - set(post)))

    # per-boss totals
    rows = {}
    for b in bosses:
        t, se = strat_total(df, df[b])
        rows[b] = {"est_total_kills": t, "se": se,
                   "ci95_low": max(t - 1.96 * se, 0), "ci95_high": t + 1.96 * se,
                   "post_2021": b in post}
    totals = pd.DataFrame(rows).T.sort_values("est_total_kills", ascending=False)

    # share of each stratum with any kills
    frac = df.groupby("stratum")[bosses].apply(lambda g: (g > 0).mean()).T
    frac.columns = [f"frac_{STRATA[h][0]}-{STRATA[h][1]}" for h in frac.columns]

    # domain estimation: flag every account, never drop any, so the weights stay valid
    df["active_post"] = (df[post].sum(axis=1) > 0).astype(float)
    n_act, se_act = strat_total(df, df["active_post"])
    k_tot, k_se = strat_total(df, df[post].sum(axis=1))
    act_by_stratum = df.groupby("stratum")["active_post"].mean()
    act_by_stratum.index = [f"{STRATA[h][0]}-{STRATA[h][1]}" for h in act_by_stratum.index]

    summary = {
        "accounts_with_post2021_kills": (n_act, 1.96 * se_act),
        "total_post2021_boss_kills": (k_tot, 1.96 * k_se),
    }

    # share of each older boss's kills that come from post-2021-active accounts
    old = [b for b in bosses if b not in post]
    share = {}
    for b in old:
        whole, _ = strat_total(df, df[b])
        part, _ = strat_total(df, df[b] * df["active_post"])
        share[b] = part / whole if whole else float("nan")
    share = pd.Series(share).sort_values(ascending=False)

    return totals, frac, summary, act_by_stratum, share

if __name__ == "__main__":
    collect()
    totals, frac, summary, act_by_stratum, share = estimate()

    print("\n--- post-2021 activity ---")
    for k, (v, pm) in summary.items():
        print(f"{k}: {v:,.0f} ± {pm:,.0f}")
    print("\nfraction of sampled accounts with post-2021 boss kills, by stratum:")
    print(act_by_stratum.round(2))

    print("\n--- top 20 bosses by estimated total kills ---")
    t = totals.copy()
    for c in ["est_total_kills", "se", "ci95_low", "ci95_high"]:
        t[c] = t[c].astype(float).map("{:,.0f}".format)
    print(t.head(20))

    print("\n--- post-2021 bosses only ---")
    print(t[t.post_2021 == True])

    print("\nshare of older-boss kills from post-2021-active accounts (top 10):")
    print(share.head(10).round(2))

    totals.join(frac).to_csv("boss_kill_estimates.csv")

2 already done, 1200 to go
10/1200
20/1200
30/1200
40/1200
50/1200
60/1200
70/1200
80/1200
90/1200
100/1200
110/1200
120/1200
130/1200
140/1200
150/1200
160/1200
170/1200
180/1200
190/1200
200/1200
210/1200
220/1200
230/1200
240/1200
250/1200
260/1200
270/1200
280/1200
290/1200
skip rank 446974: lookup for '2E3' -> HTTP 502 (https://secure.runescape.com/m=hiscore_oldschool/index_lite.json?player=2E3)
310/1200
320/1200
330/1200
340/1200
350/1200
360/1200
370/1200
380/1200
390/1200
400/1200
410/1200
420/1200
430/1200
440/1200
450/1200
460/1200
470/1200
480/1200
490/1200
500/1200
510/1200
520/1200
530/1200
540/1200
550/1200
560/1200
570/1200
580/1200
590/1200
600/1200
610/1200
620/1200
630/1200
640/1200
650/1200
660/1200
670/1200
680/1200
690/1200
700/1200
710/1200
720/1200
730/1200
740/1200
750/1200
760/1200
770/1200
780/1200
790/1200
800/1200
810/1200
820/1200
830/1200
840/1200
850/1200
860/1200
870/1200
880/1200
890/1200
900/1200
910/1200
920/1200
930/1200
940/1200
950/1200
960/1200
97

In [50]:
FILES_AB = ["boss_sample.jsonl"]      # was both A and B
FILE_C   = "boss_sample_c.jsonl"

def frame_from(files):
    """Load one or more checkpoint files into one DataFrame (drops failed lookups and repeated ranks)."""
    seen, rows = set(), []
    for fn in files:
        for line in open(fn):
            rec = json.loads(line)
            key = (rec["stratum"], rec["rank"])
            if not rec["kills"] or key in seen:
                continue
            seen.add(key)
            rows.append({"stratum": rec["stratum"], **rec["kills"]})
    return pd.DataFrame(rows).fillna(0)

def boss_estimates(df, bosses):
    return pd.DataFrame({b: strat_total(df, df[b]) for b in bosses},
                        index=["total", "se"]).T

ab = frame_from(FILES_AB)
c  = frame_from([FILE_C])

print("A+B per stratum:", ab.stratum.value_counts().sort_index().to_dict())
print("C   per stratum:", c.stratum.value_counts().sort_index().to_dict())

bosses_ab = {x for x in ab.columns if x != "stratum"}
bosses_c  = {x for x in c.columns if x != "stratum"}
common = sorted(bosses_ab & bosses_c)
print("only in A+B:", sorted(bosses_ab - bosses_c))
print("only in C:  ", sorted(bosses_c - bosses_ab))

est_ab = boss_estimates(ab, common)
est_c  = boss_estimates(c, common)

# 1) boss-by-boss agreement (the two samples are independent, so the z-test is valid)
z = (est_ab.total - est_c.total) / ((est_ab.se**2 + est_c.se**2) ** 0.5)
z = z.replace([float("inf"), float("-inf")], float("nan")).dropna()
print("\n--- per-boss z-scores, A+B vs C ---")
print(z.describe())
print("share with |z| > 2:", round((z.abs() > 2).mean(), 3))
print(z[z.abs() > 2])

# 2) headline quantities
post = sorted(b for b in POST_2021 if b in common)
def post_total(df):
    return strat_total(df, df[post].sum(axis=1))
(t1, s1), (t2, s2) = post_total(ab), post_total(c)
print(f"\npost-2021 total kills  A+B: {t1:,.0f} ± {1.96*s1:,.0f}")
print(f"                          C: {t2:,.0f} ± {1.96*s2:,.0f}")
print("z =", round((t1 - t2) / (s1**2 + s2**2) ** 0.5, 2))

# 3) pool everything for the final estimates
all_df = frame_from(FILES_AB + [FILE_C])
print("\nPooled per stratum:", all_df.stratum.value_counts().sort_index().to_dict())
final_bosses = [x for x in all_df.columns if x != "stratum"]
final = boss_estimates(all_df, final_bosses)
final["rel_se"] = final.se / final.total
final["ci95_low"] = (final.total - 1.96 * final.se).clip(lower=0)
final["ci95_high"] = final.total + 1.96 * final.se
final["nonzero_accounts"] = (all_df[final_bosses] > 0).sum()
final["post_2021"] = final.index.isin(POST_2021)
final["reliable"] = (final.nonzero_accounts >= 10) & (final.rel_se < 0.3)
final = final.sort_values("total", ascending=False)
final.to_csv("boss_kill_estimates_final.csv")
print(final.head(25).to_string(float_format=lambda v: f"{v:,.2f}"))

A+B per stratum: {0: 78, 1: 91, 2: 98, 3: 105}
C   per stratum: {0: 53, 1: 165, 2: 813, 3: 169}
only in A+B: []
only in C:   []

--- per-boss z-scores, A+B vs C ---
count   72.00
mean     0.56
std      1.01
min     -2.27
25%     -0.13
50%      0.75
75%      1.19
max      2.50
dtype: float64
share with |z| > 2: 0.097
Abyssal Sire                 2.10
Alchemical Hydra             2.08
Barrows Chests               2.36
Doom of Mokhaiotl           -2.27
Kraken                       2.50
Skotizo                      2.24
Thermonuclear Smoke Devil    2.21
dtype: float64

post-2021 total kills  A+B: 1,785,032,561 ± 350,469,920
                          C: 1,533,343,420 ± 144,486,624
z = 1.3

Pooled per stratum: {0: 131, 1: 256, 2: 911, 3: 274}
                                   total            se  rel_se       ci95_low      ci95_high  nonzero_accounts  post_2021  reliable
Kraken                    877,367,573.78 36,737,307.73    0.04 805,362,450.64 949,372,696.93               898      False

In [49]:
seen = set()
for fn in FILES_AB + [FILE_C]:
    read = failed = dup = kept = 0
    for line in open(fn):
        rec = json.loads(line)
        read += 1
        key = (rec["stratum"], rec["rank"])
        if not rec["kills"]:
            failed += 1
        elif key in seen:
            dup += 1
        else:
            seen.add(key)
            kept += 1
    print(f"{fn}: read={read} failed={failed} dup_of_earlier_file={dup} kept={kept}")

# does B's first record look like A's? (same rank = same plan)
a_first = json.loads(open(FILES_AB[0]).readline())
b_first = json.loads(open(FILES_AB[1]).readline())
print("A first:", a_first["stratum"], a_first["rank"], a_first["name"])
print("B first:", b_first["stratum"], b_first["rank"], b_first["name"])

boss_sample.jsonl: read=374 failed=2 dup_of_earlier_file=0 kept=372
boss_sample_b.jsonl: read=300 failed=0 dup_of_earlier_file=298 kept=2
boss_sample_c.jsonl: read=1200 failed=0 dup_of_earlier_file=0 kept=1200
A first: 3 1192402 Wolfsoulz
B first: 3 1192402 My_Name_Jeb


In [51]:
import numpy as np

def boot_totals(df, bosses, B=1000, seed=1):
    rng = np.random.default_rng(seed)
    groups = [df.loc[df.stratum == h, bosses].to_numpy(float) for h in range(len(STRATA))]
    Ns = [hi - lo + 1 for lo, hi in STRATA]
    out = np.zeros((B, len(bosses)))
    for i in range(B):
        tot = np.zeros(len(bosses))
        for g, N in zip(groups, Ns):
            tot += N * g[rng.integers(0, len(g), len(g))].mean(axis=0)
        out[i] = tot
    return out

bs = boot_totals(all_df, final_bosses)
final["boot_low"]  = np.percentile(bs, 2.5, axis=0)
final["boot_high"] = np.percentile(bs, 97.5, axis=0)
final["boot_rel_width"] = (final.boot_high - final.boot_low) / final.total
final["reliable"] = (final.nonzero_accounts >= 10) & (final.boot_rel_width < 0.6)

# the same for the post-2021 total
post = sorted(b for b in POST_2021 if b in final_bosses)
idx = [final_bosses.index(b) for b in post]
post_bs = bs[:, idx].sum(axis=1)
print(f"post-2021 total, bootstrap 95%: {np.percentile(post_bs, 2.5):,.0f} to {np.percentile(post_bs, 97.5):,.0f}")

cols = ["total", "ci95_low", "ci95_high", "boot_low", "boot_high", "nonzero_accounts", "reliable"]
print(final[cols].head(25).to_string(float_format=lambda v: f"{v:,.0f}"))
final.to_csv("boss_kill_estimates_final.csv")

post-2021 total, bootstrap 95%: 1,449,457,790 to 1,710,440,582
                                total    ci95_low   ci95_high    boot_low   boot_high  nonzero_accounts  reliable
Kraken                    877,367,574 805,362,451 949,372,697 138,375,446 174,996,900               898      True
Zulrah                    528,603,197 450,736,380 606,470,014 216,702,077 283,126,492              1065      True
Vorkath                   369,097,198 321,524,921 416,669,476  21,444,607  34,562,719              1071      True
Wintertodt                327,905,152 299,447,888 356,362,415 152,643,322 198,736,708              1421      True
Cerberus                  309,324,191 270,364,408 348,283,973  50,776,890  86,201,661               720      True
Thermonuclear Smoke Devil 257,235,789 223,643,896 290,827,682 216,002,783 249,568,190               691      True
Alchemical Hydra          249,219,666 216,427,698 282,011,634 144,541,568 195,693,310               545      True
Barrows Chests           

In [52]:
post_point = final.loc[post, "total"].sum()
lo, hi = np.percentile(post_bs, [2.5, 97.5])
print(f"post-2021 boss kills: {post_point:,.0f}  (95% bootstrap CI {lo:,.0f} to {hi:,.0f})")


post-2021 boss kills: 1,582,652,623  (95% bootstrap CI 1,449,457,790 to 1,710,440,582)


In [53]:
def est_accounts_with_kills(boss):
    return strat_total(all_df, (all_df[boss] > 0).astype(float))

for b in ["Zulrah", "Yama", "Araxxor"]:
    t, se = est_accounts_with_kills(b)
    print(f"{b}: estimated {t:,.0f} accounts with kills (95% CI {t-1.96*se:,.0f} to {t+1.96*se:,.0f})")
    

Zulrah: estimated 871,276 accounts with kills (95% CI 823,249 to 919,303)
Yama: estimated 196,676 accounts with kills (95% CI 173,794 to 219,559)
Araxxor: estimated 295,964 accounts with kills (95% CI 269,685 to 322,242)


Zulrah final 609,825 on page #https://secure.runescape.com/m=hiscore_oldschool/overall?category_type=1&table=91&page=24393

Yama final 185,351 on page #https://secure.runescape.com/m=hiscore_oldschool/overall?category_type=1&table=89&page=7415

Araxxor final 274,788 on page #https://secure.runescape.com/m=hiscore_oldschool/overall?category_type=1&table=23&page=10992


In [55]:
for h, (lo, hi) in enumerate(STRATA):
    g = all_df.loc[all_df.stratum == h, "Zulrah"] > 0
    print(f"stratum {h}: n={len(g)}, share with Zulrah kills={g.mean():.2f}, "
          f"est. accounts={(hi-lo+1)*g.mean():,.0f}")


stratum 0: n=131, share with Zulrah kills=0.98, est. accounts=9,847
stratum 1: n=256, share with Zulrah kills=0.98, est. accounts=87,891
stratum 2: n=911, share with Zulrah kills=0.71, est. accounts=642,151
stratum 3: n=274, share with Zulrah kills=0.13, est. accounts=131,387


In [56]:
recs = [json.loads(l) for l in open("boss_sample_c.jsonl")]
cands = [r for r in recs if r["kills"] and r["kills"].get("Zulrah", 0) > 0 and r["stratum"] in (2, 3)]
random.Random(1).shuffle(cands)

ranks = []
for r in cands[:25]:
    data = session.get(HS_JSON, params={"player": r["name"]}, timeout=30).json()
    z = next(a for a in data["activities"] if a["name"] == "Zulrah")
    ranks.append((r["stratum"], r["name"], z["score"], z["rank"]))
    time.sleep(PAUSE)

out = pd.DataFrame(ranks, columns=["stratum", "name", "kills", "zulrah_rank"])
print(out.sort_values("zulrah_rank").to_string(index=False))
print("max rank:", out.zulrah_rank.max(), "| ranks above 609,825:", (out.zulrah_rank > 609_825).sum(),
      "| unranked (-1) despite kills:", (out.zulrah_rank == -1).sum())

 stratum         name  kills  zulrah_rank
       2  Clognizance      1           -1
       2   kennydrive      1           -1
       2      uwubama      1           -1
       2     Gridleak      3           -1
       2  xImpurePure      2           -1
       2  TWERKS0S0FT      1           -1
       2      PumpPip      1           -1
       2 LogicChanger   6632         6852
       2    0xyxclean   3356        24450
       2        dicot   1232       109625
       2    Sita Koll   1088       127124
       2       Cilldo    759       181970
       2   Shiny Typh    659       203815
       2   Nether_Rod    455       260145
       2   Sir Pegger    443       264025
       2    NQG AKing    308       314071
       2     ZeusTTGs    180       379566
       2 Salad Amulet    114       428980
       2   SilasJones    111       431166
       2 daewoo lanos    111       431412
       2       Meesin    107       434569
       2   HotPotatas     47       499362
       2 Tsar Leviath     33      

In [57]:
def est_at_least(boss, T):
    return strat_total(all_df, (all_df[boss] >= T).astype(float))

TRUE_COUNT = {"Zulrah": 609_825, "Yama": 185_351}
for boss, true in TRUE_COUNT.items():
    print(f"\n{boss} (true ranked accounts: {true:,})")
    for T in [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]:
        t, se = est_at_least(boss, T)
        flag = "<-- true count inside CI" if t - 1.96*se <= true <= t + 1.96*se else ""
        print(f"  kills >= {T:>2}: {t:>10,.0f}  ({t-1.96*se:,.0f} to {t+1.96*se:,.0f}) {flag}")
        


Zulrah (true ranked accounts: 609,825)
  kills >=  1:    871,276  (823,249 to 919,303) 
  kills >=  2:    699,003  (651,356 to 746,650) 
  kills >=  3:    675,929  (628,203 to 723,656) 
  kills >=  4:    661,714  (614,822 to 708,607) 
  kills >=  5:    650,798  (604,327 to 697,269) <-- true count inside CI
  kills >=  6:    641,171  (595,590 to 686,752) <-- true count inside CI
  kills >=  8:    623,574  (578,904 to 668,244) <-- true count inside CI
  kills >= 10:    615,545  (571,350 to 659,740) <-- true count inside CI
  kills >= 15:    598,223  (555,010 to 641,437) <-- true count inside CI
  kills >= 20:    581,966  (539,781 to 624,151) <-- true count inside CI

Yama (true ranked accounts: 185,351)
  kills >=  1:    196,676  (173,794 to 219,559) <-- true count inside CI
  kills >=  2:    190,331  (167,736 to 212,926) <-- true count inside CI
  kills >=  3:    186,939  (164,521 to 209,357) <-- true count inside CI
  kills >=  4:    182,636  (160,458 to 204,813) <-- true count inside

In [62]:
TRUE_COUNT = {"Araxxor": 274790}
for boss, true in TRUE_COUNT.items():
    for T in [1, 3, 5, 10, 15, 20]:
        t, se = est_at_least(boss, T)
        print(boss, f">= {T}", f"{t:,.0f}", f"({t-1.96*se:,.0f} to {t+1.96*se:,.0f})",
              "<-- inside" if t-1.96*se <= true <= t+1.96*se else "")

Araxxor >= 1 295,964 (269,685 to 322,242) <-- inside
Araxxor >= 3 284,904 (259,835 to 309,973) <-- inside
Araxxor >= 5 276,934 (252,138 to 301,730) <-- inside
Araxxor >= 10 271,861 (247,165 to 296,557) <-- inside
Araxxor >= 15 265,439 (240,941 to 289,936) <-- inside
Araxxor >= 20 262,683 (238,266 to 287,101) <-- inside


In [63]:
import difflib

# hiscores spellings are my best guess, so the script prints any it can't find
TARGETS = {
    # raids
    "Chambers of Xeric (incl. CM)": ["Chambers of Xeric", "Chambers of Xeric: Challenge Mode"],
    "Theatre of Blood (incl. HM)":  ["Theatre of Blood", "Theatre of Blood: Hard Mode"],
    "Tombs of Amascut (incl. Expert)": ["Tombs of Amascut", "Tombs of Amascut: Expert Mode"],
    # Dagannoth Kings
    "Dagannoth Kings (all three)": ["Dagannoth Rex", "Dagannoth Prime", "Dagannoth Supreme"],
    "Dagannoth Rex": ["Dagannoth Rex"],
    "Dagannoth Prime": ["Dagannoth Prime"],
    "Dagannoth Supreme": ["Dagannoth Supreme"],
    # God Wars
    "General Graardor": ["General Graardor"],
    "Kree'Arra": ["Kree'Arra"],
    "K'ril Tsutsaroth": ["K'ril Tsutsaroth"],
    "Commander Zilyana": ["Commander Zilyana"],
    # the rest
    "Nightmare": ["Nightmare"],
    "Phosani's Nightmare": ["Phosani's Nightmare"],
    "Abyssal Sire": ["Abyssal Sire"],
    "Alchemical Hydra": ["Alchemical Hydra"],
    "Thermonuclear Smoke Devil": ["Thermonuclear Smoke Devil"],
    "Kraken": ["Kraken"],
    "Zulrah": ["Zulrah"],
    "Corporeal Beast": ["Corporeal Beast"],
}
POST_ONLY = {"Tombs of Amascut (incl. Expert)"}     # every kill is after 2021-12-09

# build one column per row of the table (multi-boss rows are summed per account)
cols, missing = {}, []
for label, members in TARGETS.items():
    found = [m for m in members if m in all_df.columns]
    for m in members:
        if m not in all_df.columns:
            missing.append((label, m, difflib.get_close_matches(m, all_df.columns, n=2)))
    if found:
        cols[label] = found
print("not found in data (label, name, closest matches):")
for x in missing:
    print("  ", x)

work = all_df.copy()
for label, found in cols.items():
    work[f"T::{label}"] = work[found].sum(axis=1)
tcols = [f"T::{l}" for l in cols]

# point estimates, normal CI, bootstrap CI
rows = {}
for l in cols:
    t, se = strat_total(work, work[f"T::{l}"])
    rows[l] = {"total": t, "se": se}
tab = pd.DataFrame(rows).T

bs = boot_totals(work, tcols)                       # the function from the earlier cell
tab["boot_low"]  = np.percentile(bs, 2.5, axis=0)
tab["boot_high"] = np.percentile(bs, 97.5, axis=0)
tab["rel_width"] = (tab.boot_high - tab.boot_low) / tab.total
tab["nonzero_accounts"] = [(work[f"T::{l}"] > 0).sum() for l in cols]
tab["period"] = ["since release (post-2021)" if l in POST_ONLY else "lifetime (includes pre-2021)" for l in cols]
tab["reliable"] = (tab.nonzero_accounts >= 10) & (tab.rel_width < 0.6)

fmt = lambda v: f"{v:,.0f}"
show = tab[["total", "boot_low", "boot_high", "nonzero_accounts", "reliable", "period"]].copy()
for c in ["total", "boot_low", "boot_high"]:
    show[c] = show[c].map(fmt)
print(show.to_string())
tab.to_csv("target_boss_estimates.csv")

not found in data (label, name, closest matches):
                                       total     boot_low    boot_high  nonzero_accounts  reliable                        period
Chambers of Xeric (incl. CM)     108,248,266   94,374,729  124,516,720               815      True  lifetime (includes pre-2021)
Theatre of Blood (incl. HM)       44,448,726   32,294,407   61,681,901               346     False  lifetime (includes pre-2021)
Tombs of Amascut (incl. Expert)   57,481,772   49,271,985   65,556,574               734      True     since release (post-2021)
Dagannoth Kings (all three)      483,415,092  432,479,561  537,935,592              1041      True  lifetime (includes pre-2021)
Dagannoth Rex                    205,055,956  186,055,768  224,132,745              1020      True  lifetime (includes pre-2021)
Dagannoth Prime                  139,300,104  123,188,180  156,670,379               885      True  lifetime (includes pre-2021)
Dagannoth Supreme                139,059,031  1

Kill totals are estimated from a stratified random sample of about 1,570 ranked accounts (four rank strata, 1 to 2,000,000), weighted to the ranked population. Figures are lifetime kill counts as of November, 13, 2026 and include kills made before December 9, 2021, except for Tombs of Amascut and the other bosses released after that date, whose totals reflect kills since release. Intervals are 95% bootstrap intervals.

In [64]:
BossKillDF = pd.read_csv("target_boss_estimates.csv")

In [67]:
tab_raw = tab.copy()          # untouched copy in case you need the original columns

tab = tab_raw.copy()
tab["Estimated total kills"]        = tab["total"].round(0).astype(int)
tab["Low end (95% range)"]          = tab["boot_low"].round(0).astype(int)
tab["High end (95% range)"]         = tab["boot_high"].round(0).astype(int)
tab["Range width (% of estimate)"]  = (tab["rel_width"] * 100).round(0).astype(int)
tab["Sampled accounts with kills"]  = tab["nonzero_accounts"].astype(int)
tab["Kills cover"]                  = tab["period"]
tab["Reliable?"]                    = tab["reliable"].map({True: "Yes", False: "No"})

tab = tab[["Estimated total kills", "Low end (95% range)", "High end (95% range)",
           "Range width (% of estimate)", "Sampled accounts with kills",
           "Reliable?", "Kills cover"]]
tab.index.name = "Boss"
tab = tab.sort_values("Estimated total kills", ascending=False)

print(tab.to_string(formatters={
    "Estimated total kills": "{:,}".format,
    "Low end (95% range)": "{:,}".format,
    "High end (95% range)": "{:,}".format,
}))
tab.to_csv("target_boss_estimates_readable.csv")

                                Estimated total kills Low end (95% range) High end (95% range)  Range width (% of estimate)  Sampled accounts with kills Reliable?                   Kills cover
Boss                                                                                                                                                                                            
Kraken                                    877,367,574         811,162,802          943,605,207                           15                          898       Yes  lifetime (includes pre-2021)
Zulrah                                    528,603,197         453,184,604          607,130,916                           29                         1065       Yes  lifetime (includes pre-2021)
Dagannoth Kings (all three)               483,415,092         432,479,561          537,935,592                           22                         1041       Yes  lifetime (includes pre-2021)
Thermonuclear Smoke Devil          

## What each column in the estimates table means

| Column | Plain meaning |
|---|---|
| `Estimated total kills` | Best estimate of the total kills for that boss across all ranked accounts. This is the headline number. |
| `se` | Standard error: how much that estimate would typically wobble from one random sample to the next. Smaller is better. It's the basis of the normal interval. |
| `Low end (95% range)` | Low end of the 95% range. If you repeated the sampling many times, the true total would fall between `boot_low` and `High end (95% range)` about 95% of the time. |
| `boot_high` | High end of that range. |
| `rel_width` | Width of the range relative to the estimate, `(boot_high - boot_low) / total`. A value of 0.30 means the range spans 30% of the estimate, roughly ±15%. Lower means more precise. |
| `nonzero_accounts` | How many of your sampled accounts have at least one kill on that boss. This is how much real data sits behind the number. A total built on 5 accounts is shaky however it's weighted. |
| `period` | What the kills cover: "lifetime" includes kills before Dec 9, 2021, and "since release" means every kill is after it. |
| `reliable` | `True` if at least 10 sampled accounts have kills **and** the range is narrower than 60% of the estimate (`rel_width < 0.6`). Those cutoffs are my rules of thumb, so adjust them if you can justify others. |

In [69]:
def var_share(df, col):
    parts = {}
    for h, (lo, hi) in enumerate(STRATA):
        v = df.loc[df.stratum == h, col].to_numpy(float)
        n, N = len(v), hi - lo + 1
        parts[h] = N**2 * v.var(ddof=1) / n * (1 - n / N)
    s = sum(parts.values())
    return {h: round(p / s, 2) for h, p in parts.items()}

wide = ["Corporeal Beast", "Theatre of Blood (incl. HM)", "Nightmare",
        "Phosani's Nightmare", "Kree'Arra"]
for b in wide:
    print(b, var_share(work, f"T::{b}"))

Corporeal Beast {0: np.float64(0.01), 1: np.float64(0.01), 2: np.float64(0.97), 3: np.float64(0.0)}
Theatre of Blood (incl. HM) {0: np.float64(0.03), 1: np.float64(0.06), 2: np.float64(0.91), 3: np.float64(0.0)}
Nightmare {0: np.float64(0.03), 1: np.float64(0.06), 2: np.float64(0.89), 3: np.float64(0.02)}
Phosani's Nightmare {0: np.float64(0.05), 1: np.float64(0.08), 2: np.float64(0.84), 3: np.float64(0.02)}
Kree'Arra {0: np.float64(0.02), 1: np.float64(0.2), 2: np.float64(0.78), 3: np.float64(0.0)}


In [70]:
# one combined target: each wide boss scaled by its own total so they count equally
work["wide_target"] = sum(work[f"T::{b}"] / tab_raw.loc[b, "total"] for b in wide)
alloc = neyman(work, "wide_target", 1500)
print(alloc)            # accounts per stratum for a 1,500-account batch

{0: 71, 1: 175, 2: 1213, 3: 42}


In [72]:
N_BY_STRATUM = alloc
SEED = 123                       # unused so far
CKPT = "boss_sample_d.jsonl"     # new file
plan = make_plan()
print(plan[0], "already used?", plan[0] in load_done())   # should print False
collect()

(0, 6524) already used? True
1501 already done, 0 to go


In [73]:
POOL = ["boss_sample.jsonl", "boss_sample_c.jsonl", "boss_sample_d.jsonl"]   # use your actual d file name(s); never B

seen = set()
for fn in POOL:
    read = failed = dup = kept = 0
    for line in open(fn):
        rec = json.loads(line)
        read += 1
        key = (rec["stratum"], rec["rank"])
        if not rec["kills"]:
            failed += 1
        elif key in seen:
            dup += 1
        else:
            seen.add(key)
            kept += 1
    print(f"{fn}: read={read} failed={failed} dup={dup} kept={kept}")

boss_sample.jsonl: read=374 failed=2 dup=0 kept=372
boss_sample_c.jsonl: read=1200 failed=0 dup=0 kept=1200
boss_sample_d.jsonl: read=1501 failed=0 dup=1 kept=1500


In [85]:
from bs4 import BeautifulSoup

# ---------- fix: back off and retry when the rankings page has no table ----------
class PageUnavailable(Exception):
    pass

PAUSE = 3.0      # slower pace than before

def names_on_page(page):
    r = get(HS_PAGE, {"table": 0, "page": page})
    if r is None:
        return []
    soup = BeautifulSoup(r.text, "lxml")
    trs = soup.select("tr")
    name_idx = None
    for tr in trs:
        labels = [c.get_text(strip=True).lower() for c in tr.find_all(["td", "th"])]
        if "name" in labels:
            name_idx = labels.index("name")
            break
    if name_idx is None:
        title = soup.title.get_text(strip=True) if soup.title else None
        raise PageUnavailable(f"no Name header (page title: {title})")
    names = []
    for tr in trs:
        if tr.find("th"):
            continue
        cells = tr.find_all(["td", "th"])
        if len(cells) <= name_idx:
            continue
        t = cells[name_idx].get_text(" ", strip=True).replace("\xa0", " ").strip()
        if t:
            names.append(t)
    return names

def collect():
    plan, done = make_plan(), load_done()
    todo = [p for p in plan if p not in done]
    print(f"{len(done)} already done, {len(todo)} to go")
    with open(CKPT, "a") as f:
        for i, (h, rank) in enumerate(todo, 1):
            kills = name = None
            for attempt in range(4):
                try:
                    name = name_at_rank(rank)
                    time.sleep(PAUSE)
                    if not name:
                        raise RuntimeError(f"no name at rank {rank}")
                    kills = boss_kills(name)
                    time.sleep(PAUSE)
                    break
                except PageUnavailable as e:
                    wait = 300 * (attempt + 1)            # 5, 10, 15, 20 minutes
                    print(f"rank {rank}: {e}; waiting {wait // 60} min (attempt {attempt + 1}/4)")
                    time.sleep(wait)
                except Exception as e:
                    print(f"skip rank {rank}: {e}")
                    time.sleep(PAUSE)                     # pause even on failure
                    break
            else:
                raise RuntimeError("rankings pages keep coming back without a table; "
                                   "stop here and resume later")
            if kills is None:
                continue                                  # skipped; retried on the next run
            f.write(json.dumps({"stratum": h, "rank": rank, "name": name, "kills": kills}) + "\n")
            f.flush()
            if i % 10 == 0:
                print(f"{i}/{len(todo)}")

# ---------- your original cell, unchanged apart from the comments ----------
def build_table(files, B=1000):
    df = frame_from(files)
    cols = {l: [m for m in ms if m in df.columns] for l, ms in TARGETS.items()}
    cols = {l: f for l, f in cols.items() if f}
    for l, f in cols.items():
        df[f"T::{l}"] = df[f].sum(axis=1)
    tcols = [f"T::{l}" for l in cols]
    t = pd.DataFrame({l: strat_total(df, df[f"T::{l}"]) for l in cols}, index=["total", "se"]).T
    bs = boot_totals(df, tcols, B=B)
    t["boot_low"] = np.percentile(bs, 2.5, axis=0)
    t["boot_high"] = np.percentile(bs, 97.5, axis=0)
    t["rel_width"] = (t.boot_high - t.boot_low) / t.total
    t["nonzero_accounts"] = [(df[c] > 0).sum() for c in tcols]
    return df, t

POOL = ["boss_sample.jsonl", "boss_sample_c.jsonl"]      # not B, which repeats A's ranks
df_cur, t_cur = build_table(POOL)

df_cur["avg_rel"] = sum(df_cur[f"T::{l}"] / t_cur.loc[l, "total"] for l in t_cur.index)
N_BY_STRATUM = neyman(df_cur, "avg_rel", 1000)
print(N_BY_STRATUM)                  # should match what you got before: {0: 48, 1: 126, 2: 789, 3: 37}

SEED = 201
CKPT = "boss_sample_d1.jsonl"
plan = make_plan()
print(plan[0], "already used?", plan[0] in load_done())   # True is expected when RESUMING this batch; False for a brand-new one
collect()                            # resumable; stops with an error if the site keeps refusing

POOL.append(CKPT)
df_cur, t_cur = build_table(POOL)
print("accounts:", len(df_cur), "| bosses under 15% width:", (t_cur.rel_width < 0.15).sum(), "of", len(t_cur))
print((t_cur.rel_width * 100).round(0).sort_values().to_string())

{0: 48, 1: 126, 2: 789, 3: 37}
(2, 350758) already used? True
468 already done, 532 to go
10/532
20/532
rank 488275: no Name header (page title: Error - Old School RuneScape); waiting 5 min (attempt 1/4)
30/532
40/532
50/532
60/532
rank 90255: no Name header (page title: Error - Old School RuneScape); waiting 5 min (attempt 1/4)
70/532
80/532
90/532
100/532
rank 16011: no Name header (page title: Error - Old School RuneScape); waiting 5 min (attempt 1/4)
110/532
120/532
130/532
140/532
rank 257573: no Name header (page title: Error - Old School RuneScape); waiting 5 min (attempt 1/4)
150/532
160/532
170/532
180/532
rank 320233: no Name header (page title: Error - Old School RuneScape); waiting 5 min (attempt 1/4)
190/532
200/532
210/532
220/532
rank 732587: no Name header (page title: Error - Old School RuneScape); waiting 5 min (attempt 1/4)
230/532
240/532
250/532
260/532
rank 537715: no Name header (page title: Error - Old School RuneScape); waiting 5 min (attempt 1/4)
270/532
280/5

In [86]:
import glob
print(sorted(glob.glob("boss_sample*.jsonl")))      # confirm the exact file names first

POOL = ["boss_sample.jsonl", "boss_sample_c.jsonl", "boss_sample_d.jsonl", "boss_sample_d1.jsonl"]

seen = set()
for fn in POOL:
    read = failed = dup = kept = 0
    for line in open(fn):
        rec = json.loads(line)
        read += 1
        key = (rec["stratum"], rec["rank"])
        if not rec["kills"]:
            failed += 1
        elif key in seen:
            dup += 1
        else:
            seen.add(key)
            kept += 1
    print(f"{fn}: read={read} failed={failed} dup={dup} kept={kept}")
print("total kept:", len(seen))

['boss_sample.jsonl', 'boss_sample_b.jsonl', 'boss_sample_c.jsonl', 'boss_sample_d.jsonl', 'boss_sample_d1.jsonl', 'boss_sample_pooled.jsonl']
boss_sample.jsonl: read=374 failed=2 dup=0 kept=372
boss_sample_c.jsonl: read=1200 failed=0 dup=0 kept=1200
boss_sample_d.jsonl: read=1501 failed=0 dup=1 kept=1500
boss_sample_d1.jsonl: read=1000 failed=0 dup=4 kept=996
total kept: 4068


In [87]:
for fn in sorted(glob.glob("boss_sample*.jsonl")):
    first = json.loads(open(fn).readline())
    print(fn, "first record:", first["stratum"], first["rank"])

boss_sample.jsonl first record: 3 1192402
boss_sample_b.jsonl first record: 3 1192402
boss_sample_c.jsonl first record: 3 1262262
boss_sample_d.jsonl first record: 0 6524
boss_sample_d1.jsonl first record: 2 906101
boss_sample_pooled.jsonl first record: 3 1192402


In [80]:
# check on osrs hiscore server 
r = session.get(HS_PAGE, params={"table": 0, "page": (771417 - 1) // 25 + 1}, timeout=30)
print(r.status_code, r.url)
print(r.headers.get("Retry-After"), r.headers.get("Server"))
print(r.text[:600])

200 https://secure.runescape.com/m=hiscore_oldschool/overall?table=0&page=30857
None cloudflare

<!doctype html>
<!--[if lt IE 7]><html class="no-js lt-ie10 lt-ie9 lt-ie8 lt-ie7" lang="en"><![endif]-->
<!--[if (IE 7)&!(IEMobile)]><html class="no-js lt-ie10 lt-ie9 lt-ie8" lang="en"><![endif]-->
<!--[if (IE 8)&!(IEMobile)]><html class="no-js lt-ie10 lt-ie9" lang="en"><![endif]-->
<!--[if (IE 9)&!(IEMobile)]><html class="no-js lt-ie10" lang="en"><![endif]-->
<!--[if gt IE 9]><!--><!-- x --> <html class="no-js" lang="en"> <!--<![endif]-->
<head>
<link href="https://www.runescape.com/css/rsp777_basic-128.css" rel="stylesheet" type="text/css" media="all">
<script>
    window.dataLayer = window


In [ ]:
def get(url, params=None):
    for attempt in range(6):
        r = session.get(url, params=params, timeout=30)
        if r.status_code == 404:
            return None
        looks_blocked = r.status_code in (403, 429, 500, 502, 503, 504) or (
            r.status_code == 200 and "<th" not in r.text and "index_lite" not in str(url))
        if looks_blocked:
            wait = int(r.headers.get("Retry-After", 60 * (attempt + 1)))
            print(f"possible throttling (HTTP {r.status_code}), waiting {wait}s")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"gave up on {url} {params} (last status {r.status_code})")

In [88]:
df_cur, t_cur = build_table(POOL)
print("accounts:", len(df_cur), "| per stratum:", df_cur.stratum.value_counts().sort_index().to_dict())

t = t_cur.copy()
t["period"] = ["since release (post-2021)" if l in POST_ONLY else "lifetime (includes pre-2021)" for l in t.index]
WIDTH_TARGET = 0.15
t["reliable"] = (t.nonzero_accounts >= 10) & (t.rel_width < WIDTH_TARGET)

final_tab = pd.DataFrame({
    "Estimated total kills":       t["total"].round(0).astype(int),
    "Low end (95% range)":         t["boot_low"].round(0).astype(int),
    "High end (95% range)":        t["boot_high"].round(0).astype(int),
    "Range width (% of estimate)": (t["rel_width"] * 100).round(0).astype(int),
    "Sampled accounts with kills": t["nonzero_accounts"].astype(int),
    "Within 15% target?":          t["reliable"].map({True: "Yes", False: "No"}),
    "Kills cover":                 t["period"],
}).sort_values("Estimated total kills", ascending=False)
final_tab.index.name = "Boss"

print(final_tab.to_string(formatters={c: "{:,}".format for c in
      ["Estimated total kills", "Low end (95% range)", "High end (95% range)"]}))
final_tab.to_csv("target_boss_estimates_final.csv")

accounts: 4068 | per stratum: {0: 248, 1: 557, 2: 2910, 3: 353}
                                Estimated total kills Low end (95% range) High end (95% range)  Range width (% of estimate)  Sampled accounts with kills Within 15% target?                   Kills cover
Boss                                                                                                                                                                                                     
Kraken                                    890,919,312         848,763,669          937,422,754                           10                         2405                Yes  lifetime (includes pre-2021)
Zulrah                                    492,575,878         450,324,073          544,960,278                           19                         2878                 No  lifetime (includes pre-2021)
Dagannoth Kings (all three)               479,125,894         441,106,372          521,304,002                           17     

In [90]:
import shutil, datetime, json

# tier column, then save the final table
tier = lambda w: "≤15%" if w <= 0.15 else "15–25%" if w <= 0.25 else ">25%"
final_tab["Precision tier"] = t_cur.rel_width.map(tier).reindex(final_tab.index)
final_tab.to_csv("target_boss_estimates_FINAL.csv")

# keep copies of the raw samples you used (never B or the old pooled file)
POOL = ["boss_sample.jsonl", "boss_sample_c.jsonl", "boss_sample_d.jsonl", "boss_sample_d1.jsonl"]
import os; os.makedirs("final_data", exist_ok=True)
for fn in POOL:
    shutil.copy(fn, os.path.join("final_data", fn))

meta = {
    "analysis_date": str(datetime.date.today()),
    "files_pooled": POOL,
    "excluded_files": {"boss_sample_b.jsonl": "repeated sample A's ranks", "boss_sample_pooled.jsonl": "merged copy of earlier files"},
    "accounts_used": int(len(df_cur)),
    "accounts_per_stratum": {str(k): int(v) for k, v in df_cur.stratum.value_counts().sort_index().items()},
    "strata_ranks": STRATA,
    "bootstrap_resamples": 1000,
    "post_2021_bosses": sorted(POST_2021),
}
json.dump(meta, open("final_data/metadata.json", "w"), indent=2)
print(meta["accounts_used"], meta["accounts_per_stratum"])

4068 {'0': 248, '1': 557, '2': 2910, '3': 353}


In [91]:
def est_at_least_df(df, boss, T):
    return strat_total(df, (df[boss] >= T).astype(float))

TRUE_COUNT = {"Zulrah": 609_825, "Yama": 185_351}      # add Araxxor's real last rank
for boss, true in TRUE_COUNT.items():
    for T in [1, 5, 10, 15]:
        t, se = est_at_least_df(df_cur, boss, T)
        print(boss, f">= {T}", f"{t:,.0f}", f"({t-1.96*se:,.0f} to {t+1.96*se:,.0f})",
              "<-- inside" if t-1.96*se <= true <= t+1.96*se else "")

Zulrah >= 1 854,308 (816,697 to 891,919) 
Zulrah >= 5 624,042 (588,720 to 659,363) <-- inside
Zulrah >= 10 584,428 (551,718 to 617,138) <-- inside
Zulrah >= 15 565,072 (533,185 to 596,959) 
Yama >= 1 186,236 (171,968 to 200,503) <-- inside
Yama >= 5 173,903 (159,958 to 187,848) <-- inside
Yama >= 10 165,564 (151,838 to 179,289) 
Yama >= 15 160,735 (147,144 to 174,327) 


In [92]:
post = sorted(b for b in POST_2021 if b in df_cur.columns)
print(len(post), "post-2021 bosses in the data")
# bootstrap needs a combined column
df_cur["POST2021_TOTAL"] = df_cur[post].sum(axis=1)
tot, se = strat_total(df_cur, df_cur["POST2021_TOTAL"])
bs = boot_totals(df_cur, ["POST2021_TOTAL"], B=1000)
print(f"post-2021 boss kills: {tot:,.0f}  (95% bootstrap CI {np.percentile(bs,2.5):,.0f} to {np.percentile(bs,97.5):,.0f})")

25 post-2021 bosses in the data
post-2021 boss kills: 1,539,380,119  (95% bootstrap CI 1,465,443,698 to 1,608,828,861)


In [ ]:
Zulrah final 609,825 on page #https://secure.runescape.com/m=hiscore_oldschool/overall?category_type=1&table=91&page=24393

Yama final 185,351 on page #https://secure.runescape.com/m=hiscore_oldschool/overall?category_type=1&table=89&page=7415

Araxxor final 274,788 on page #https://secure.runescape.com/m=hiscore_oldschool/overall?category_type=1&table=23&page=10992

In [94]:
LAST_ROW_KILLS = {"Zulrah": 5, "Yama": 5}   # replace with the real numbers
TRUE_COUNT = {"Zulrah": 609927, "Yama": 185445}                     # reread these too, ranks shift

for boss, T in LAST_ROW_KILLS.items():
    t, se = est_at_least_df(df_cur, boss, T)
    true = TRUE_COUNT[boss]
    print(boss, f">= {T}:", f"{t:,.0f}", f"({t-1.96*se:,.0f} to {t+1.96*se:,.0f})",
          "<-- inside" if t-1.96*se <= true <= t+1.96*se else "<-- OUTSIDE")

Zulrah >= 5: 624,042 (588,720 to 659,363) <-- inside
Yama >= 5: 173,903 (159,958 to 187,848) <-- inside


To check the sampling weights, the estimated number of accounts with at least 5 kills was compared with the number of ranked players on the Zulrah and Yama boss hiscores (609,825 and 185,351 at the time of checking). Both known counts fell within the 95% bootstrap intervals (Zulrah 588,720 to 659,363; Yama 159,958 to 187,848).

Write the limitations: ranked accounts only (about the top 2 million), ironmen and other account types included, lifetime kills except for Tombs of Amascut, sampling error only, and the precision tiers, because several bosses are wider than 15%.

In [95]:
import os, json
print(os.listdir("final_data"))
print(open("final_data/metadata.json").read())

['boss_sample_d.jsonl', 'boss_sample.jsonl', 'metadata.json', 'boss_sample_c.jsonl', 'boss_sample_d1.jsonl']
{
  "analysis_date": "2026-11-14",
  "files_pooled": [
    "boss_sample.jsonl",
    "boss_sample_c.jsonl",
    "boss_sample_d.jsonl",
    "boss_sample_d1.jsonl"
  ],
  "excluded_files": {
    "boss_sample_b.jsonl": "repeated sample A's ranks",
    "boss_sample_pooled.jsonl": "merged copy of earlier files"
  },
  "accounts_used": 4068,
  "accounts_per_stratum": {
    "0": 248,
    "1": 557,
    "2": 2910,
    "3": 353
  },
  "strata_ranks": [
    [
      1,
      10000
    ],
    [
      10001,
      100000
    ],
    [
      100001,
      1000000
    ],
    [
      1000001,
      2000000
    ]
  ],
  "bootstrap_resamples": 1000,
  "post_2021_bosses": [
    "Amoxliatl",
    "Araxxor",
    "Artio",
    "Brutus",
    "Calvar'ion",
    "Doom of Mokhaiotl",
    "Duke Sucellus",
    "Lunar Chests",
    "Mad Angel",
    "Maggot King",
    "Nex",
    "Phantom Muspah",
    "Scurrius",
  

In [98]:
meta = json.load(open("final_data/metadata.json"))

meta["collection_dates"] = {
    "boss_sample.jsonl (A)":      "2026-11-13",
    "boss_sample_c.jsonl (C)":    "2026-11-13",
    "boss_sample_d.jsonl (D)":    "2026-11-14",
    "boss_sample_d1.jsonl (D1)":  "2026-11-14",
}
meta["seeds"] = {"A": 42, "C": 98, "D1": 201, "D": 123,}
meta["validation"] = {
    "method": "estimated accounts with >=5 kills vs ranked players on the boss hiscores",
    "Zulrah": {"known_count": 609825, "estimate": 624042, "ci95": [588720, 659363]},
    "Yama":   {"known_count": 185351, "estimate": 173903, "ci95": [159958, 187848]},
    "date_checked": "2026-11-14",
}
meta["post_2021_total_kills"] = {"estimate": 1539380119, "ci95_bootstrap": [1465443698, 1608828861]}
meta["notes"] = "Ranked accounts only; ironmen and other account types included; kills are lifetime except Tombs of Amascut and post-2021 bosses."

with open("final_data/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

{
  "analysis_date": "2026-11-14",
  "files_pooled": [
    "boss_sample.jsonl",
    "boss_sample_c.jsonl",
    "boss_sample_d.jsonl",
    "boss_sample_d1.jsonl"
  ],
  "excluded_files": {
    "boss_sample_b.jsonl": "repeated sample A's ranks",
    "boss_sample_pooled.jsonl": "merged copy of earlier files"
  },
  "accounts_used": 4068,
  "accounts_per_stratum": {
    "0": 248,
    "1": 557,
    "2": 2910,
    "3": 353
  },
  "strata_ranks": [
    [
      1,
      10000
    ],
    [
      10001,
      100000
    ],
    [
      100001,
      1000000
    ],
    [
      1000001,
      2000000
    ]
  ],
  "bootstrap_resamples": 1000,
  "post_2021_bosses": [
    "Amoxliatl",
    "Araxxor",
    "Artio",
    "Brutus",
    "Calvar'ion",
    "Doom of Mokhaiotl",
    "Duke Sucellus",
    "Lunar Chests",
    "Mad Angel",
    "Maggot King",
    "Nex",
    "Phantom Muspah",
    "Scurrius",
    "Shellbane Gryphon",
    "Sol Heredit",
    "Spindel",
    "The Fractured Archive",
    "The Hueycoatl",
  

In [99]:
import datetime
for fn in ["boss_sample.jsonl", "boss_sample_c.jsonl", "boss_sample_d.jsonl", "boss_sample_d1.jsonl"]:
    print(fn, datetime.datetime.fromtimestamp(os.path.getmtime(f"final_data/{fn}")).date())

boss_sample.jsonl 2026-11-14
boss_sample_c.jsonl 2026-11-14
boss_sample_d.jsonl 2026-11-14
boss_sample_d1.jsonl 2026-11-14


In [100]:
SINK_TARGETS = {
    # raids
    "Chambers of Xeric (incl. CM)": ["Chambers of Xeric", "Chambers of Xeric: Challenge Mode"],
    "Theatre of Blood (incl. HM)": ["Theatre of Blood", "Theatre of Blood: Hard Mode"],
    "Tombs of Amascut (incl. Expert)": ["Tombs of Amascut", "Tombs of Amascut: Expert Mode"],
    # God Wars Dungeon / Godsword shards
    "General Graardor": ["General Graardor"],
    "Kree'Arra": ["Kree'Arra"],
    "K'ril Tsutsaroth": ["K'ril Tsutsaroth"],
    "Commander Zilyana": ["Commander Zilyana"],
    # Dagannoth Kings
    "Dagannoth Kings (all three)": ["Dagannoth Rex", "Dagannoth Prime", "Dagannoth Supreme"],
    # Nightmare
    "Nightmare": ["Nightmare"],
    "Phosani's Nightmare": ["Phosani's Nightmare"],
    # Slayer bosses (candidates: confirm against your list)
    "Abyssal Sire": ["Abyssal Sire"],
    "Alchemical Hydra": ["Alchemical Hydra"],
    "Thermonuclear Smoke Devil": ["Thermonuclear Smoke Devil"],
    "Kraken": ["Kraken"],
    "Cerberus": ["Cerberus"],
    "Grotesque Guardians": ["Grotesque Guardians"],
    # bosses from your earlier list that aren't in a category above
    "Zulrah": ["Zulrah"],
    "Corporeal Beast": ["Corporeal Beast"],
}
# Zenyte shard: I'm not sure which sources it covers. If they are demonic gorillas or tormented
# demons, the hiscores don't track those as bosses, so the sample can't estimate them.

df_chk = frame_from(POOL)
for label, members in SINK_TARGETS.items():
    gone = [m for m in members if m not in df_chk.columns]
    if gone:
        print("not in hiscores data:", label, gone, difflib.get_close_matches(gone[0], df_chk.columns, n=2))

TARGETS = SINK_TARGETS           # build_table reads this global
POST_ONLY = {"Tombs of Amascut (incl. Expert)"}
df_sink, t_sink = build_table(POOL)

t = t_sink.copy()
t["period"] = ["since release (post-2021)" if l in POST_ONLY else "lifetime (includes pre-2021)" for l in t.index]
sink_tab = pd.DataFrame({
    "Estimated total kills":       t["total"].round(0).astype(int),
    "Low end (95% range)":         t["boot_low"].round(0).astype(int),
    "High end (95% range)":        t["boot_high"].round(0).astype(int),
    "Range width (% of estimate)": (t["rel_width"] * 100).round(0).astype(int),
    "Sampled accounts with kills": t["nonzero_accounts"].astype(int),
    "Kills cover":                 t["period"],
}).sort_values("Estimated total kills", ascending=False)
sink_tab.index.name = "Boss"
print(sink_tab.to_string(formatters={c: "{:,}".format for c in
      ["Estimated total kills", "Low end (95% range)", "High end (95% range)"]}))
sink_tab.to_csv("sink_list_boss_estimates.csv")

                                Estimated total kills Low end (95% range) High end (95% range)  Range width (% of estimate)  Sampled accounts with kills                   Kills cover
Boss                                                                                                                                                                                  
Kraken                                    890,919,312         848,763,669          937,422,754                           10                         2405  lifetime (includes pre-2021)
Zulrah                                    492,575,878         450,324,073          544,960,278                           19                         2878  lifetime (includes pre-2021)
Dagannoth Kings (all three)               479,125,894         441,106,372          521,304,002                           17                         2777  lifetime (includes pre-2021)
Cerberus                                  291,627,844         271,151,796          31

In [101]:
sink_tab

,Estimated total kills,Low end (95% range),High end (95% range),Range width (% of estimate),Sampled accounts with kills,Kills cover
Boss,,,,,,
Kraken,890919312,848763669,937422754,10,2405,lifetime (includes pre-2021)
Zulrah,492575878,450324073,544960278,19,2878,lifetime (includes pre-2021)
Dagannoth Kings (all three),479125894,441106372,521304002,17,2777,lifetime (includes pre-2021)
Cerberus,291627844,271151796,312756186,14,1829,lifetime (includes pre-2021)
Thermonuclear Smoke Devil,275059723,252773269,299631436,17,1758,lifetime (includes pre-2021)
Alchemical Hydra,250637807,232269451,269425011,15,1346,lifetime (includes pre-2021)
General Graardor,176709574,156022765,201029649,25,2473,lifetime (includes pre-2021)
Abyssal Sire,151509008,141475818,162205763,14,1807,lifetime (includes pre-2021)
Grotesque Guardians,111542460,102455121,121112259,17,2006,lifetime (includes pre-2021)


In [121]:
import pandas as pd
from osrs_drop_rates_fix import scrape_drop_rates

items = pd.read_csv("ItemSinkList.csv")  # find the column that holds the item names
item_col = "item"
items[item_col] = items[item_col].astype(str).str.strip()

drops = scrape_drop_rates(items, item_col=item_col)
drops.to_csv("item_drop_rates.csv", index=False)
drops

No item sources found for: ['arcane spirit shield', 'burning claws', 'dragon hunter lance', 'godsword']


,item,wiki_page,category,source,level,quantity,rarity_raw,drop_rate,one_in_x
0,abyssal bludgeon,Unsired,Item sources,Abyssal Sire,350,1,1/100,0.01,100.00
1,ancestral hat,ancestral hat,Item sources,Ancient chest Normal Mode,N/A,1,4/60,0.07,15.00
2,ancestral hat,ancestral hat,Item sources,Ancient chest Challenge Mode,N/A,1,4/56,0.07,14.00
3,ancestral robe bottom,ancestral robe bottom,Item sources,Ancient chest Normal Mode,N/A,1,4/60,0.07,15.00
4,ancestral robe bottom,ancestral robe bottom,Item sources,Ancient chest Challenge Mode,N/A,1,4/56,0.07,14.00
...,...,...,...,...,...,...,...,...,...
119,zamorakian spear,zamorakian spear,Item sources,Balfrug Kreeyath,151,1,"3/16,129",0.00,"5,376.33"
120,zamorakian spear,zamorakian spear,Item sources,Zakl'n Gritch,142,1,"3/16,129",0.00,"5,376.33"
121,zamorakian spear,zamorakian spear,Item sources,Tstanon Karlak,145,1,"3/16,129",0.00,"5,376.33"
122,zaryte crossbow,Nihil horn,Item sources,Nex,1001,1,1/258,0.00,258.00


In [123]:
# 1) per-boss kills table: every hiscores boss gets its own row, same columns as sink_tab
df_k = frame_from(POOL)
bosses = [c for c in df_k.columns if c != "stratum"]
kt = pd.DataFrame({b: strat_total(df_k, df_k[b]) for b in bosses}, index=["total", "se"]).T
bs = boot_totals(df_k, bosses, B=1000)
kills_tab = pd.DataFrame({
    "Estimated total kills": kt["total"],
    "Low end (95% range)":   np.percentile(bs, 2.5, axis=0),
    "High end (95% range)":  np.percentile(bs, 97.5, axis=0),
}, index=kt.index)
kills_tab.index.name = "Boss"
print(len(kills_tab), "bosses in kills_tab")

72 bosses in kills_tab


In [124]:
# 2) drops = kills x drop_rate x quantity, summed over sources, grouped by item
KCOL, LCOL, HCOL = "Estimated total kills", "Low end (95% range)", "High end (95% range)"

# wiki source name (lowercase) -> kills_tab row. Names that already match a hiscores boss map automatically.
SOURCE_MAP = {
    "ancient chest normal mode":    "Chambers of Xeric",
    "ancient chest challenge mode": "Chambers of Xeric: Challenge Mode",
    # add the rest from the unmatched list below, e.g. your Theatre of Blood and Tombs of Amascut chest names
}

# multiply a source's rate by this when the listed rate is NOT per completion
RATE_MULT = {
    # "ancient chest normal mode": <chance of a unique per completion>,
}

def parse_qty(q):
    s = str(q).replace(",", "").replace("–", "-").strip()
    try:
        return float(s)
    except ValueError:
        if "-" in s:
            try:
                lo, hi = [float(x) for x in s.split("-")[:2]]
                return (lo + hi) / 2
            except ValueError:
                pass
    return 1.0

def estimate_drops(drops, kills):
    d = drops.copy()
    n0 = len(d)
    d = d.drop_duplicates(subset=["item", "source", "rarity_raw", "quantity"])
    print(f"dropped {n0 - len(d)} duplicate rows")
    d["drop_rate"] = pd.to_numeric(d["drop_rate"], errors="coerce")
    d = d.dropna(subset=["drop_rate"])
    d["qty"] = d["quantity"].map(parse_qty)
    d["key"] = d["source"].astype(str).str.lower().str.strip()
    lookup = {k.lower(): k for k in kills.index}
    d["boss"] = d["key"].map(lambda k: SOURCE_MAP.get(k) or lookup.get(k))
    d["mult"] = d["key"].map(RATE_MULT).fillna(1.0)

    unmatched = d.loc[d.boss.isna(), "source"].value_counts()
    print(f"\n{len(unmatched)} sources have no kills estimate (their drops are NOT counted):")
    print(unmatched.to_string())

    m = d.dropna(subset=["boss"]).copy()
    m["w"] = m.drop_rate * m.qty * m.mult
    for col, name in ((KCOL, "est"), (LCOL, "low"), (HCOL, "high")):
        m[name] = m.boss.map(kills[col]) * m.w

    out = m.groupby("item")[["est", "low", "high"]].sum()
    out["range_width_%"] = ((out.high - out.low) / out.est * 100).round(0)
    out["sources_counted"] = m.groupby("item").size()
    out["sources_listed"] = d.groupby("item").size()
    out["lower_bound"] = out.sources_counted < out.sources_listed
    out.index.name = "item"
    return out.rename(columns={"est": "Estimated drops", "low": "Low end", "high": "High end"}) \
              .sort_values("Estimated drops", ascending=False), unmatched

item_drops, unmatched = estimate_drops(drops, kills_tab)    # use your drop-rate dataframe's name
print(item_drops.round(0).to_string())
item_drops.round(0).to_csv("sink_item_drop_estimates.csv")

dropped 0 duplicate rows

25 sources have no kills estimate (their drops are NOT counted):
source
The Nightmare                   7
Chest (Tombs of Amascut)        7
Flight Kilisa                   3
Wingman Skree                   3
Flockleader Geerin              3
Monumental chest Normal Mode    3
Monumental chest Hard Mode      3
Sergeant Strongstack            3
Sergeant Steelwill              3
Sergeant Grimspike              3
Bandosian guard                 2
Armadylean guard                2
Cave kraken                     2
Dark beast                      1
Night beast                     1
Lizardman shaman                1
Smoke devil                     1
Nuclear smoke devil             1
Starlight                       1
Bree                            1
Growler                         1
Ancient chest                   1
Balfrug Kreeyath                1
Zakl'n Gritch                   1
Tstanon Karlak                  1
                         Estimated drops       Low e

In [126]:
KCOL, LCOL, HCOL = "Estimated total kills", "Low end (95% range)", "High end (95% range)"

combo = {
    "Chambers of Xeric (all modes)": ["Chambers of Xeric", "Chambers of Xeric: Challenge Mode"],
    "Tombs of Amascut (all modes)":  ["Tombs of Amascut", "Tombs of Amascut: Expert Mode"],
}
cdf = df_k.copy()
for name, ms in combo.items():
    cdf[name] = cdf[ms].sum(axis=1)
cn = list(combo)
ct = pd.DataFrame({n: strat_total(cdf, cdf[n]) for n in cn}, index=["total", "se"]).T
cbs = boot_totals(cdf, cn, B=1000)
add = pd.DataFrame({KCOL: ct.total,
                    LCOL: np.percentile(cbs, 2.5, axis=0),
                    HCOL: np.percentile(cbs, 97.5, axis=0)}, index=cn)
kills_tab2 = pd.concat([kills_tab, add])
kills_tab2.index.name = "Boss"

SOURCE_MAP = {
    "ancient chest normal mode":    "Chambers of Xeric",
    "ancient chest challenge mode": "Chambers of Xeric: Challenge Mode",
    "ancient chest":                "Chambers of Xeric (all modes)",
    "monumental chest normal mode": "Theatre of Blood",
    "monumental chest hard mode":   "Theatre of Blood: Hard Mode",
    "chest (tombs of amascut)":     "Tombs of Amascut (all modes)",
    "the nightmare":                "Nightmare",
}

item_drops, unmatched = estimate_drops(drops, kills_tab2)
print(item_drops.round(0).to_string())
item_drops.round(0).to_csv("sink_item_drop_estimates.csv")

dropped 0 duplicate rows

20 sources have no kills estimate (their drops are NOT counted):
source
Flight Kilisa           3
Wingman Skree           3
Flockleader Geerin      3
Sergeant Strongstack    3
Sergeant Steelwill      3
Sergeant Grimspike      3
Bandosian guard         2
Armadylean guard        2
Cave kraken             2
Dark beast              1
Night beast             1
Lizardman shaman        1
Smoke devil             1
Nuclear smoke devil     1
Starlight               1
Bree                    1
Growler                 1
Balfrug Kreeyath        1
Zakl'n Gritch           1
Tstanon Karlak          1
                               Estimated drops       Low end      High end  range_width_%  sources_counted  sources_listed  lower_bound
item                                                                                                                                   
dexterous prayer scroll          24,303,706.00 21,724,163.00 27,215,288.00          23.00                2    

In [128]:
item_drops

,Estimated drops,Low end,High end,range_width_%,sources_counted,sources_listed,lower_bound
item,,,,,,,
dexterous prayer scroll,"24,303,705.79","21,724,162.53","27,215,287.70",23.00,2,2,False
arcane prayer scroll,"24,303,705.79","21,724,162.53","27,215,287.70",23.00,2,2,False
avernic defender hilt,"18,828,988.58","15,815,772.30","22,382,114.41",35.00,2,2,False
lightbearer,"16,054,889.41","14,752,100.17","17,434,087.48",17.00,1,1,False
osmumten's fang,"16,054,889.41","14,752,100.17","17,434,087.48",17.00,1,1,False
ancestral robe bottom,"7,096,324.51","6,340,561.42","7,950,187.02",23.00,2,2,False
ancestral robe top,"7,096,324.51","6,340,561.42","7,950,187.02",23.00,2,2,False
ancestral hat,"7,096,324.51","6,340,561.42","7,950,187.02",23.00,2,2,False
twisted buckler,"7,096,324.51","6,340,561.42","7,950,187.02",23.00,2,2,False


In [129]:
chk = drops.copy()
chk["w"] = pd.to_numeric(chk["drop_rate"], errors="coerce") * chk["quantity"].map(parse_qty)
by_src = chk.groupby("source")["w"].sum().sort_values(ascending=False)
print(by_src.loc[[s for s in by_src.index if "chest" in s.lower() or "nightmare" in s.lower()]])

source
Chest (Tombs of Amascut)       1.00
Ancient chest Normal Mode      0.92
Ancient chest Challenge Mode   0.91
Monumental chest Normal Mode   0.58
Monumental chest Hard Mode     0.56
Ancient chest                  0.03
The Nightmare                  0.01
Phosani's Nightmare            0.01
Name: w, dtype: float64


In [130]:
import io, contextlib

def p_unique_cox(points):
    # per the wiki as I remember it: points / 867,600, capped at 0.657. Verify before relying on it.
    return min(points / 867_600, 0.657)

cox_items = ["twisted bow", "dexterous prayer scroll", "arcane prayer scroll", "ancestral robe top",
             "elder maul", "dinh's bulwark", "twisted buckler"]

def cox_sensitivity(avg_points_list):
    rows = {}
    for pts in avg_points_list:
        p = p_unique_cox(pts)
        RATE_MULT.clear()
        RATE_MULT.update({"ancient chest normal mode": p, "ancient chest challenge mode": p})
        with contextlib.redirect_stdout(io.StringIO()):          # hides the unmatched-source printout
            out, _ = estimate_drops(drops, kills_tab2)
        rows[f"{pts:,} pts (P={p:.3f})"] = out["Estimated drops"]
    return pd.DataFrame(rows).loc[cox_items]

print(cox_sensitivity([10_000, 20_000, 30_000, 40_000]).round(0).to_string())

                         10,000 pts (P=0.012)  20,000 pts (P=0.023)  30,000 pts (P=0.035)  40,000 pts (P=0.046)
item                                                                                                           
twisted bow                         40,896.00             81,793.00            122,689.00            163,585.00
dexterous prayer scroll            280,126.00            560,251.00            840,377.00          1,120,503.00
arcane prayer scroll               280,126.00            560,251.00            840,377.00          1,120,503.00
ancestral robe top                  81,793.00            163,585.00            245,378.00            327,170.00
elder maul                          40,896.00             81,793.00            122,689.00            163,585.00
dinh's bulwark                      61,344.00            122,689.00            184,033.00            245,378.00
twisted buckler                     81,793.00            163,585.00            245,378.00            327

In [131]:
base_pts = 30_000                                   # whichever value you run with
lo_pts, mid_pts, hi_pts = 15_000, 30_000, 45_000    # replace with your own range

p = p_unique_cox(mid_pts)
RATE_MULT.clear()
RATE_MULT.update({"ancient chest normal mode": p, "ancient chest challenge mode": p})
with contextlib.redirect_stdout(io.StringIO()):
    out, _ = estimate_drops(drops, kills_tab2)

raid = out.loc[cox_items, ["Estimated drops", "Low end", "High end"]].copy()
raid.columns = ["Estimate (mid points)", "Sampling low", "Sampling high"]
raid["Assumption low"]  = raid["Estimate (mid points)"] * lo_pts / mid_pts
raid["Assumption high"] = raid["Estimate (mid points)"] * hi_pts / mid_pts
print(raid.round(0).to_string())

                         Estimate (mid points)  Sampling low  Sampling high  Assumption low  Assumption high
item                                                                                                        
twisted bow                         122,689.00    109,622.00     137,451.00       61,344.00       184,033.00
dexterous prayer scroll             840,377.00    751,181.00     941,054.00      420,189.00     1,260,566.00
arcane prayer scroll                840,377.00    751,181.00     941,054.00      420,189.00     1,260,566.00
ancestral robe top                  245,378.00    219,245.00     274,903.00      122,689.00       368,067.00
elder maul                          122,689.00    109,622.00     137,451.00       61,344.00       184,033.00
dinh's bulwark                      184,033.00    164,434.00     206,177.00       92,017.00       276,050.00
twisted buckler                     245,378.00    219,245.00     274,903.00      122,689.00       368,067.00


In [132]:
raid

,Estimate (mid points),Sampling low,Sampling high,Assumption low,Assumption high
item,,,,,
twisted bow,"122,688.87","109,622.43","137,451.37","61,344.44","184,033.31"
dexterous prayer scroll,"840,377.10","751,181.28","941,054.21","420,188.55","1,260,565.65"
arcane prayer scroll,"840,377.10","751,181.28","941,054.21","420,188.55","1,260,565.65"
ancestral robe top,"245,377.75","219,244.86","274,902.73","122,688.87","368,066.62"
elder maul,"122,688.87","109,622.43","137,451.37","61,344.44","184,033.31"
dinh's bulwark,"184,033.31","164,433.65","206,177.05","92,016.66","276,049.97"
twisted buckler,"245,377.75","219,244.86","274,902.73","122,688.87","368,066.62"


In [133]:
def tob_mult(team_size, deaths_factor=1.0):
    return {"monumental chest normal mode": deaths_factor / (9.1 * team_size),
            "monumental chest hard mode":   deaths_factor / (7.7 * team_size)}

tob_items = ["avernic defender hilt", "sanguinesti staff (uncharged)", "scythe of vitur (uncharged)"]
rows = {}
for N in [1, 2, 3, 4, 5]:                      # illustrative team sizes; replace with your assumption
    RATE_MULT.clear()
    RATE_MULT.update({"ancient chest normal mode": p_unique_cox(30_000),       # your CoX central value
                      "ancient chest challenge mode": p_unique_cox(30_000)})
    RATE_MULT.update(tob_mult(N, deaths_factor=1.0))      # 1.0 = ceiling; lower it for typical deaths
    with contextlib.redirect_stdout(io.StringIO()):
        out, _ = estimate_drops(drops, kills_tab2)
    rows[f"team of {N}"] = out["Estimated drops"]
tob = pd.DataFrame(rows).loc[tob_items]
print(tob.round(0).to_string())

                                 team of 1    team of 2  team of 3  team of 4  team of 5
item                                                                                    
avernic defender hilt         2,116,148.00 1,058,074.00 705,383.00 529,037.00 423,230.00
sanguinesti staff (uncharged)   539,954.00   269,977.00 179,985.00 134,989.00 107,991.00
scythe of vitur (uncharged)     269,977.00   134,989.00  89,992.00  67,494.00  53,995.00


In [134]:
ITEM_MULT = {}     # (item name lowercase, source key) -> extra multiplier, defined once outside the function

# in estimate_drops, directly after the line that sets d["mult"]:
d["mult"] = d["mult"] * [ITEM_MULT.get((str(i).lower(), k), 1.0) for i, k in zip(d["item"], d["key"])]

NameError: name 'd' is not defined

In [135]:
P_UNIQUE_TOA  = ...   # chance of a unique per player completion, from the wiki's formula and your assumed contribution and level
F_LT50        = ...   # share of completions at raid level 0-49
F_LT150       = ...   # share at raid level 0-149

low50  = (1 - F_LT50)  + F_LT50  / 50       # fang and lightbearer
low150 = (1 - F_LT150) + F_LT150 / 50       # ward, masori, shadow

src = "chest (tombs of amascut)"
RATE_MULT[src] = P_UNIQUE_TOA
for it in ["osmumten's fang", "lightbearer"]:
    ITEM_MULT[(it, src)] = low50
for it in ["elidinis' ward", "masori mask (f)", "masori body (f)", "masori chaps (f)", "tumeken's shadow (uncharged)"]:
    ITEM_MULT[(it, src)] = low150

TypeError: unsupported operand type(s) for -: 'int' and 'ellipsis'